In [ ]:
import torch
import numpy as np
import re
import sys
import io

# Force UTF-8 output for Hindi/Telugu characters
''' sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8') '''

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- ALIGNMENT ENGINE ---

class WordAligner:
    def __init__(self, model_name='sentence-transformers/LaBSE', device=None):
        if device is None:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        else:
            self.device = device
        print(f"Loading alignment model {model_name} on {self.device}...")
        self.model = SentenceTransformer(model_name).to(self.device)

    def align(self, src_text, tgt_text, src_tokens, tgt_tokens, threshold=0.15, rescue_threshold=0.1):
        """
        Produce word alignment links.
        """
        if not src_tokens or not tgt_tokens:
            return set()

        # Encode tokens
        src_emb = self.model.encode(src_tokens, convert_to_tensor=True).cpu().numpy()
        tgt_emb = self.model.encode(tgt_tokens, convert_to_tensor=True).cpu().numpy()

        # Similarity matrix
        sim_mat = cosine_similarity(src_emb, tgt_emb)

        links = set()

        # 1. Mutual Best Match (MBM)
        for i in range(len(src_tokens)):
            for j in range(len(tgt_tokens)):
                is_best_for_src = (j == np.argmax(sim_mat[i, :]))
                is_best_for_tgt = (i == np.argmax(sim_mat[:, j]))

                if is_best_for_src and is_best_for_tgt and sim_mat[i, j] > threshold:
                    links.add((i, j))

        # 2. Rescue Orphans (Many-to-Many support)
        linked_src = set(l[0] for l in links)
        linked_tgt = set(l[1] for l in links)

        for i in range(len(src_tokens)):
            if i not in linked_src:
                j = np.argmax(sim_mat[i, :])
                if sim_mat[i, j] > rescue_threshold:
                    links.add((i, j))

        for j in range(len(tgt_tokens)):
            if j not in linked_tgt:
                i = np.argmax(sim_mat[:, j])
                if sim_mat[i, j] > rescue_threshold:
                    links.add((i, j))

        return links

# --- EVALUATION LOGIC ---

def calculate_aer(gold_s, gold_p, predicted):
    a = set(predicted)
    s = set(gold_s)
    p = set(gold_p) | s

    intersection_s = a.intersection(s)
    intersection_p = a.intersection(p)

    num = len(intersection_s) + len(intersection_p)
    den = len(a) + len(s)

    if den == 0: return 0.0
    return 1.0 - (num / den)

def get_tokens(text):
    return re.sub(r'[।\?\!\,\(\)"]', '', text).strip().split()

# --- DATASET (100 PAIRS) ---

HINDI_TELUGU_DATASET = [
    ("मैं स्कूल जा रहा हूँ।", "నేను పాఠశాలకు వెళ్తున్నాను।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 2), (5, 3)}),
    ("आज मौसम बहुत अच्छा है।", "ఈ రోజు వాతావరణం చాలా బాగుంది।", {(0, 0), (0, 1), (1, 2), (2, 3), (3, 4)}, {(4, 4), (5, 5)}),
    ("क्या आप मेरी मदद कर सकते हैं?", "మీరు నాకు సహాయం చేయగలరా?", {(1, 0), (2, 1), (3, 2), (4, 3)}, {(0, 3), (5, 3), (6, 3), (7, 4)}),
    ("मुझे भूख लग रही है।", "నాకు ఆకలిగా ఉంది।", {(0, 0), (1, 1)}, {(2, 1), (3, 2), (4, 2), (5, 3)}),
    ("वह मेरा सबसे अच्छा दोस्त है।", "అతను నా ప్రాణ స్నేహితుడు।", {(0, 0), (1, 1), (4, 3)}, {(2, 2), (3, 2), (5, 3), (6, 4)}),
    ("भारत एक महान देश है।", "భారతదేశం ఒక గొప్ప దేశం।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("कल छुट्टी है।", "రేపు సెలవు।", {(0, 0), (1, 1)}, {(2, 1), (3, 2)}),
    ("आपको यहाँ देखकर खुशी हुई।", "మిమ్మల్ని ఇక్కడ చూడటం సంతోషంగా ఉంది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 5)}),
    ("पानी जीवन के लिए आवश्यक है।", "నీరు జీవించడానికి అవసరం।", {(0, 0), (1, 1), (4, 2)}, {(2, 1), (3, 1), (5, 2), (6, 3)}),
    ("सूर्य पूर्व में उदय होता है।", "సూర్యుడు తూర్పున ఉదయిస్తాడు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 2), (6, 3)}),
    ("मैं आपसे कल मिलूँगा।", "నేను మిమ్మల్ని రేపు కలుస్తాను।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4)}),
    ("यह फिल्म बहुत डरावनी है।", "ఈ సినిమా చాలా భయానకంగా ఉంది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 5)}),
    ("मुझे कॉफी पीना पसंद है।", "నాకు కాఫీ తాగడం ఇష్టం।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("बच्चे बगीचे में खेल रहे हैं।", "పిల్లలు తోటలో ఆడుకుంటున్నారు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 2), (6, 3)}),
    ("दिल्ली भारत की राजधानी है।", "ఢిల్లీ భారతదేశ రాజధాని।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 3)}),
    ("ईमानदारी सबसे अच्छी नीति है।", "నిజాయితీ ఉత్తమ విధానం।", {(0, 0), (3, 2)}, {(1, 1), (2, 1), (4, 2), (5, 3)}),
    ("कृपया दरवाजा बंद करें।", "దయచేసి తలుపు మూయండి।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),
    ("वह गाना बहुत मधुर है।", "ఆ పాట చాలా మధురంగా ఉంది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 5)}),
    ("सफलता के लिए कड़ी मेहनत जरूरी है।", "విజయానికి కష్టపడటం అవసరం।", {(0, 0), (4, 1), (5, 2)}, {(1, 0), (2, 0), (3, 1), (6, 2), (7, 3)}),
    ("हम सब भारतीय हैं।", "మనమందరం భారతీయులం।", {(0, 0), (1, 0), (2, 1)}, {(3, 1), (4, 2)}),

    # Batch 2: Nature, Food, Travel
    ("बगीचे में फूल खिल रहे हैं।", "తోటలో పూలు పూస్తున్నాయి।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),
    ("नदी घाटी के बीच से बहती है।", "నది లోయ గుండా ప్రవహిస్తుంది।", {(0, 0), (1, 1), (4, 3)}, {(2, 1), (3, 1), (5, 4)}),
    ("मुझे गर्मियों में आम खाना पसंद है।", "నాకు వేసవిలో మామిడి పండ్లు తినడం ఇష్టం।", {(0, 0), (3, 2), (4, 3), (5, 4)}, {(1, 1), (2, 1), (6, 5)}),
    ("वह हमारे लिए स्वादिष्ट खाना बना रही है।", "ఆమె మా కోసం రుచికరమైన ఆహారాన్ని వండుతోంది।", {(0, 0), (2, 2), (3, 3), (4, 4)}, {(1, 1), (5, 4), (6, 5)}),
    ("आज ट्रेन देर से आ रही है।", "రైలు ఈ రోజు ఆలస్యంగా ఉంది।", {(0, 1), (1, 0), (2, 2)}, {(3, 3), (4, 4), (5, 4)}),
    ("मैं ताज महल देखना चाहता हूँ।", "నేను తాజ్ మహల్ దర్శించాలనుకుంటున్నాను।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 3)}),
    ("खाना खाने के बाद यह दवा लें।", "భోజనం తర్వాత ఈ మందు తీసుకోండి।", {(0, 3), (1, 0), (2, 1), (4, 2), (5, 4)}, {(3, 1), (6, 4)}),
    ("डॉक्टर ने उसे आराम करने की सलाह दी।", "వైద్యుడు అతనికి విశ్రాంతి తీసుకోమని సలహా ఇచ్చాడు।", {(0, 0), (2, 1), (3, 2), (6, 4)}, {(1, 0), (4, 2), (5, 3), (7, 5)}),
    ("मोबाइल फोन ने हमारी जिंदगी बदल दी है।", "మొబైల్ ఫోన్లు మన జీవితాలను మార్చాయి।", {(0, 0), (1, 1), (3, 3), (4, 4)}, {(2, 2), (5, 5), (6, 5)}),
    ("इंटरनेट ज्ञान का एक बड़ा स्रोत है।", "ఇంటర్నెట్ జ్ఞానానికి ఒక మూలం।", {(0, 0), (1, 1), (5, 4)}, {(2, 1), (3, 2), (4, 3), (6, 5)}),

    # Batch 3: More variations
    ("किताबें पढ़ना अच्छी आदत है।", "పుస్తకాలు చదవడం మంచి అలవాటు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("क्या आप अंग्रेजी बोल सकते हैं?", "మీరు ఇంగ్లీష్ మాట్లాడగలరా?", {(1, 0), (2, 1), (3, 2)}, {(0, 2), (4, 2), (5, 3)}),
    ("हिमालय दुनिया का सबसे ऊँचा पहाड़ है।", "హిమాలయాలు ప్రపంచంలోనే ఎత్తైన పర్వతం।", {(0, 0), (1, 1), (4, 3), (5, 4)}, {(2, 1), (3, 2), (6, 5)}),
    ("पेड़ हमें ऑक्सीजन देते हैं।", "చెట్లు మనకు ఆక్సిజన్ ఇస్తాయి।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("समय बहुत कीमती होता है।", "సమయం చాలా విలువైనది।", {(0, 0), (1, 1), (2, 2)}, {(3, 3), (4, 3)}),
    ("मेहनत कभी बेकार नहीं जाती।", "కష్టపడటం ఎప్పుడూ వృథా పోదు।", {(0, 0), (3, 3), (4, 4)}, {(1, 1), (2, 2), (5, 4)}),
    ("हम कल फिल्म देखने जाएंगे।", "మనం రేపు సినిమా చూడటానికి వెళ్తాము।", {(0, 0), (1, 1), (2, 2), (4, 4)}, {(3, 3), (5, 5)}),
    ("आज बहुत ठंड है।", "ఈ రోజు చాలా చలిగా ఉంది।", {(0, 0), (1, 1), (2, 3)}, {(3, 4)}),
    ("वह बहुत सुंदर चित्र बनाती है।", "ఆమె చాలా అందమైన చిత్రాలను గీస్తుంది।", {(0, 0), (2, 2), (3, 3), (4, 4)}, {(1, 1), (5, 4)}),
    ("सत्य की हमेशा जीत होती है।", "నిజం ఎప్పుడూ గెలుస్తుంది।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),

    # Batch 4: Everyday life and objects
    ("मेरे पास एक काला कुत्ता है।", "నా దగ్గర ఒక నల్ల కుక్క ఉంది।", {(0, 0), (1, 1), (3, 3), (4, 4)}, {(2, 2), (5, 5)}),
    ("दूध सेहत के लिए अच्छा है।", "పాలు ఆరోగ్యానికి మంచిది।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3), (5, 3)}),
    ("चिड़िया आसमान में उड़ रही है।", "పక్షి ఆకాశంలో ఎగురుతోంది।", {(0, 0), (1, 1), (2, 1), (3, 3)}, {(4, 3), (5, 4)}),
    ("यह घर बहुत बड़ा और सुंदर है।", "ఈ ఇల్లు చాలా పెద్దది మరియు అందంగా ఉంది।", {(0, 0), (1, 1), (3, 3), (4, 4), (5, 5)}, {(2, 2), (6, 6)}),
    ("मुझे नीले रंग की कमीज चाहिए।", "నాకు నీలం రంగు చొక్కా కావాలి।", {(0, 0), (1, 1), (2, 2), (4, 3), (5, 4)}, {(3, 2), (6, 5)}),
    ("वह रोज सुबह दौड़ने जाता है।", "అతను ప్రతిరోజూ ఉదయం పరుగెత్తడానికి వెళ్తాడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 4)}),
    ("यहाँ धूम्रपान मना है।", "ఇక్కడ ధూమపానం నిషేధించబడింది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("गाड़ी बहुत तेज चल रही है।", "బండి చాలా వేగంగా వెళ్తోంది।", {(0, 0), (2, 2), (3, 3)}, {(1, 1), (4, 3), (5, 4)}),
    ("मैं संगीत सुनना पसंद करता हूँ।", "నేను సంగీతం వినడం ఇష్టపడతాను।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("मेरे पिता एक किसान हैं।", "మా తండ్రి ఒక రైతు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3)}),

    # Batch 5: Abstract and variations
    ("अंधेरे के बाद रोशनी आती है।", "చీకటి తర్వాత వెలుగు వస్తుంది।", {(0, 0), (1, 1), (3, 2), (4, 3)}, {(2, 1), (5, 3)}),
    ("क्षमा करना एक महान गुण है।", "క్షమించడం ఒక గొప్ప గుణం।", {(0, 0), (2, 1), (3, 2), (4, 3)}, {(1, 1), (5, 4)}),
    ("साहस से डरो मत।", "ధైర్యంతో భయపడవద్దు।", {(0, 0), (2, 1)}, {(1, 0), (3, 2)}),
    ("दोस्ती दुनिया का सबसे अनमोल रिश्ता है।", "స్నేహం ప్రపంచంలోనే అతి విలువైన బంధం।", {(0, 0), (1, 1), (4, 3), (5, 4)}, {(2, 1), (3, 2), (6, 5)}),
    ("आज रात चाँद चमक रहा है।", "ఈ రాత్రి చంద్రుడు ప్రకాశిస్తున్నాడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("वह हमेशा सच बोलता है।", "అతను ఎప్పుడూ నిజం చెబుతాడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("जीवन एक संघर्ष है।", "జీవితం ఒక పోరాటం।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("हमें बड़ों का सम्मान करना चाहिए।", "మనం పెద్దలను గౌరవించాలి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 3)}),
    ("ज्ञान सबसे शक्तिशाली हथियार है।", "జ్ఞానం అత్యంత శక్తివంతమైన ఆయుధం।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3)}),
    ("अपनी मेहनत पर भरोसा रखें।", "నీ కష్టాన్ని నమ్ముకో।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2)}),

    # Batch 6: More Travel and Places
    ("स्टेशन यहाँ से दूर है।", "స్టేషన్ ఇక్కడి నుండి దూరంగా ఉంది।", {(0, 0), (1, 1), (3, 3)}, {(2, 2), (4, 3), (5, 4)}),
    ("बस स्टैंड कहाँ है?", "బస్టాండ్ ఎక్కడ ఉంది?", {(0, 0), (1, 0), (2, 1)}, {(3, 2)}),
    ("मुझे टिकट खरीदना है।", "నేను టికెట్ కొనాలి।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("अपनी सीट पर बैठ जाइए।", "మీ సీటులో కూర్చోండి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2)}),
    ("हवाई जहाज आसमान में उड़ रहा है।", "విమానం ఆకాశంలో ఎగురుతోంది।", {(0, 0), (1, 0), (2, 1), (4, 2)}, {(3, 1), (5, 2), (6, 3)}),
    ("रास्ता बहुत ऊबड़-खाबड़ है।", "దారి చాలా గతుకులుగా ఉంది।", {(0, 0), (1, 1), (2, 2)}, {(3, 3), (4, 3)}),
    ("पहाड़ों की सुंदरता अद्भुत है।", "పర్వతాల అందం అద్భుతంగా ఉంది।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),
    ("समुद्र की लहरें ऊँची हैं।", "సముద్రపు అలలు ఎత్తుగా ఉన్నాయి।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),
    ("शहर में बहुत भीड़ है।", "నగరంలో చాలా రద్దీగా ఉంది।", {(0, 0), (2, 2)}, {(1, 0), (3, 2), (4, 3)}),
    ("गाँव की शांति पसंद है।", "గ్రామపు ప్రశాంతత ఇష్టం।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 2)}),

    # Batch 7: Food and Health
    ("मुझे प्यास लगी है।", "నాకు దాహంగా ఉంది।", {(0, 0), (1, 1)}, {(2, 2), (3, 2)}),
    ("ताजा फल खाओ।", "తాజా పండ్లు తిను।", {(0, 0), (1, 1), (2, 2)}, {}),
    ("सब्जियां सेहत के लिए अच्छी होती हैं।", "కూరగాయలు ఆరోగ్యానికి మంచివి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3), (5, 3)}),
    ("पानी उबाल कर पिएं।", "నీటిని మరిగించి తాగండి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2)}),
    ("योगा करने से मन शांत रहता है।", "యోగా చేయడం వల్ల మనస్సు ప్రశాంతంగా ఉంటుంది।", {(0, 0), (1, 1), (3, 3), (4, 4)}, {(2, 2), (5, 5), (6, 5)}),
    ("हाथ धोकर खाना खाएं।", "చేతులు కడుక్కుని భోజనం చేయండి।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("ज्यादा चीनी मत खाओ।", "ఎక్కువ చక్కెర తినవద్దు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {}),
    ("नींद पूरी होनी चाहिए।", "నిద్ర పూర్తి కావాలి।", {(0, 0), (1, 1)}, {(2, 2), (3, 2)}),
    ("स्वच्छता बहुत जरूरी है।", "పరిశుభ్రత చాలా అవసరం।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("वह अस्पताल जा रहा है।", "అతను ఆసుపత్రికి వెళ్తున్నాడు।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),

    # Batch 8: Education and Work
    ("स्कूल समय पर जाओ।", "పాఠశాలకు సమయానికి వెళ్ళు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1)}),
    ("शिक्षक पाठ पढ़ा रहे हैं।", "ఉపాధ్యాయుడు పాఠం చెబుతున్నారు।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),
    ("परीक्षा की तैयारी करो।", "పరీక్షకు సిద్ధమవ్వండి।", {(0, 0), (2, 1)}, {(1, 0), (3, 1)}),
    ("सवाल का जवाब दो।", "ప్రశ్నకు సమాధానం ఇవ్వండి।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 2)}),
    ("लिखना और पढ़ना सीखो।", "రాయడం మరియు చదవడం నేర్చుకోండి।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("वह दफ्तर में काम करता है।", "అతను కార్యాలయంలో పని చేస్తాడు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3), (5, 4)}),
    ("ईमेल भेज दो।", "ఈమెయిల్ పంపండి।", {(0, 0), (1, 1), (2, 1)}, {}),
    ("मीटिंग कल होगी।", "సమావేశం రేపు జరుగుతుంది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("मेरा कंप्यूटर खराब है।", "నా కంప్యూటర్ పాడైపోయింది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("नई चीजें सीखना अच्छा है।", "కొత్త విషయాలు నేర్చుకోవడం మంచిది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),

    # Batch 9: Relationships and Feelings
    ("मेरी माँ बहुत प्यार करती है।", "మా అమ్మ చాలా ప్రేమిస్తుంది।", {(0, 0), (1, 1), (3, 3)}, {(2, 2), (4, 3)}),
    ("भाई-बहन साथ खेलते हैं।", "అన్నదమ్ములు అక్కాచెల్లెళ్లు కలిసి ఆడుకుంటారు।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 2)}),
    ("वह मेरा सच्चा मित्र है।", "అతను నా నిజమైన స్నేహితుడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("बच्चे शोर मचा रहे हैं।", "పిల్లలు అల్లరి చేస్తున్నారు।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),
    ("मुझे गुस्सा आ रहा है।", "నాకు కోపం వస్తోంది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 2)}),
    ("डरने की कोई बात नहीं।", "భయపడాల్సిన అవసరం లేదు।", {(0, 0), (3, 2)}, {(1, 0), (2, 1), (4, 2)}),
    ("खुश रहना सबसे जरूरी है।", "సంతోషంగా ఉండటం అత్యంత ముఖ్యం।", {(0, 0), (1, 1), (3, 3)}, {(2, 2), (4, 3)}),
    ("वह उदास क्यों है?", "అతను ఎందుకు విచారంగా ఉన్నాడు?", {(0, 0), (1, 1), (2, 2)}, {(3, 3), (4, 3)}),
    ("सबका भला करो।", "అందరికీ మంచి చేయి।", {(0, 0), (1, 1), (2, 2)}, {}),
    ("हम सब एक हैं।", "మనమందరం ఒక్కటే।", {(0, 0), (1, 0), (2, 1)}, {(3, 1)})
]

# --- PIPELINE INTEGRATION ---

def run_evaluation(align_func):
    total_aer = 0
    print(f"{'ID':<4} | {'Sentence (HI)':<35} | {'AER':<7}")
    print("-" * 55)

    results = []
    for i, (hi, te, s, p) in enumerate(HINDI_TELUGU_DATASET):
        hi_tokens = get_tokens(hi)
        te_tokens = get_tokens(te)

        predicted = align_func(hi, te, hi_tokens, te_tokens)
        aer = calculate_aer(s, p, predicted)
        total_aer += aer
        results.append((i+1, hi, aer))

        if i < 10 or i > 90: # Show first and last few
            display_hi = hi[:32] + "..." if len(hi) > 32 else hi
            print(f"{i+1:<4} | {display_hi:<35} | {aer:.4f}")

            # Print mappings
            mapping_strs = []
            for (si, ti) in sorted(predicted):
                if si < len(hi_tokens) and ti < len(te_tokens):
                    mapping_strs.append(f"{hi_tokens[si]}→{te_tokens[ti]}")
            print(f"     Mappings: {' | '.join(mapping_strs[:8])}")
            if len(mapping_strs) > 8:
                print(f"     ... and {len(mapping_strs)-8} more")
            print("-" * 55)
        elif i == 10:
            print("... | ...                                 | ...")
            print("-" * 55)

    avg_aer = total_aer / len(HINDI_TELUGU_DATASET)
    print(f"Final Average AER on 100 Sentences: {avg_aer:.4f}")
    return avg_aer

if __name__ == "__main__":
    print("Initializing Consolidated Hindi-Telugu Word Alignment Pipeline...")
    aligner = WordAligner()

    def align_wrapper(hi, te, hi_tokens, te_tokens):
        return aligner.align(hi, te, hi_tokens, te_tokens)

    run_evaluation(align_wrapper)


Initializing Consolidated Hindi-Telugu Word Alignment Pipeline...
Loading alignment model sentence-transformers/LaBSE on cpu...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

ID   | Sentence (HI)                       | AER    
-------------------------------------------------------
1    | मैं स्कूल जा रहा हूँ।               | 0.1250
     Mappings: मैं→నేను | स्कूल→పాఠశాలకు | जा→వెళ్తున్నాను | रहा→వెళ్తున్నాను | हूँ→నేను
-------------------------------------------------------
2    | आज मौसम बहुत अच्छा है।              | 0.2000
     Mappings: आज→రోజు | मौसम→వాతావరణం | बहुत→చాలా | अच्छा→బాగుంది | है→ఈ
-------------------------------------------------------
3    | क्या आप मेरी मदद कर सकते हैं?       | 0.2727
     Mappings: क्या→చేయగలరా | आप→మీరు | मेरी→నాకు | मदद→సహాయం | कर→మీరు | सकते→చేయగలరా | हैं→మీరు
-------------------------------------------------------
4    | मुझे भूख लग रही है।                 | 0.1429
     Mappings: मुझे→నాకు | भूख→ఆకలిగా | लग→ఉంది | रही→ఉంది | है→ఉంది
-------------------------------------------------------
5    | वह मेरा सबसे अच्छा दोस्त है।        | 0.4000
     Mappings: वह→అతను | मेरा→నా | सबसे→నా | अच्छा→స్నేహితుడు | दोस्त→స్నేహిత

In [ ]:
import torch
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from torch.nn.functional import softmax

class UnsupervisedAligner:
    def __init__(self, model_name='sentence-transformers/LaBSE'):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = SentenceTransformer(model_name).to(self.device)
        self.tokenizer = self.model.tokenizer

    def _get_contextual_vectors(self, tokens):
        inputs = self.tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.model[0].auto_model(**inputs, output_hidden_states=True)
            embeddings = outputs.hidden_states[8][0]

        word_ids = inputs.word_ids(batch_index=0)
        final_embs = []
        for i in range(len(tokens)):
            indices = [j for j, w_id in enumerate(word_ids) if w_id == i]
            final_embs.append(embeddings[indices].mean(dim=0) if indices else torch.zeros(embeddings.shape[-1]).to(self.device))
        return torch.stack(final_embs)

    def align_and_evaluate(self, hi_text, te_text, threshold=0.02):
        hi_toks = re.sub(r'[।\?\!\,\(\)"]', '', hi_text).strip().split()
        te_toks = re.sub(r'[।\?\!\,\(\)"]', '', te_text).strip().split()
        if not hi_toks or not te_toks: return [], 0.0

        hi_vecs = self._get_contextual_vectors(hi_toks)
        te_vecs = self._get_contextual_vectors(te_toks)
        sim_mat = torch.matmul(hi_vecs, te_vecs.T)

        s_hi = softmax(sim_mat * 10.0, dim=1)
        s_te = softmax(sim_mat * 10.0, dim=0)
        mutual_conf = (s_hi * s_te).cpu().numpy()

        links = [(hi_toks[i], te_toks[j], mutual_conf[i, j])
                 for i in range(len(hi_toks)) for j in range(len(te_toks))
                 if mutual_conf[i, j] > threshold]

        avg_conf = np.mean([l[2] for l in links]) if links else 0
        coverage = len(links) / max(len(hi_toks), len(te_toks))
        return links, (avg_conf + coverage) / 2

# --- DATASET ---
# (I'm using your 100-sentence structure)
HINDI_TELUGU_DATASET = [
    ("मैं स्कूल जा रहा हूँ।", "నేను పాఠశాలకు వెళ్తున్నాను।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 2), (5, 3)}),
    ("आज मौसम बहुत अच्छा है।", "ఈ రోజు వాతావరణం చాలా బాగుంది।", {(0, 0), (0, 1), (1, 2), (2, 3), (3, 4)}, {(4, 4), (5, 5)}),
    ("क्या आप मेरी मदद कर सकते हैं?", "మీరు నాకు సహాయం చేయగలరా?", {(1, 0), (2, 1), (3, 2), (4, 3)}, {(0, 3), (5, 3), (6, 3), (7, 4)}),
    ("मुझे भूख लग रही है।", "నాకు ఆకలిగా ఉంది।", {(0, 0), (1, 1)}, {(2, 1), (3, 2), (4, 2), (5, 3)}),
    ("वह मेरा सबसे अच्छा दोस्त है।", "అతను నా ప్రాణ స్నేహితుడు।", {(0, 0), (1, 1), (4, 3)}, {(2, 2), (3, 2), (5, 3), (6, 4)}),
    ("भारत एक महान देश है।", "భారతదేశం ఒక గొప్ప దేశం।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("कल छुट्टी है।", "రేపు సెలవు।", {(0, 0), (1, 1)}, {(2, 1), (3, 2)}),
    ("आपको यहाँ देखकर खुशी हुई।", "మిమ్మల్ని ఇక్కడ చూడటం సంతోషంగా ఉంది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 5)}),
    ("पानी जीवन के लिए आवश्यक है।", "నీరు జీవించడానికి అవసరం।", {(0, 0), (1, 1), (4, 2)}, {(2, 1), (3, 1), (5, 2), (6, 3)}),
    ("सूर्य पूर्व में उदय होता है।", "సూర్యుడు తూర్పున ఉదయిస్తాడు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 2), (6, 3)}),
    ("मैं आपसे कल मिलूँगा।", "నేను మిమ్మల్ని రేపు కలుస్తాను।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4)}),
    ("यह फिल्म बहुत डरावनी है।", "ఈ సినిమా చాలా భయానకంగా ఉంది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 5)}),
    ("मुझे कॉफी पीना पसंद है।", "నాకు కాఫీ తాగడం ఇష్టం।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("बच्चे बगीचे में खेल रहे हैं।", "పిల్లలు తోటలో ఆడుకుంటున్నారు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 2), (6, 3)}),
    ("दिल्ली भारत की राजधानी है।", "ఢిల్లీ భారతదేశ రాజధాని।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 3)}),
    ("ईमानदारी सबसे अच्छी नीति है।", "నిజాయితీ ఉత్తమ విధానం।", {(0, 0), (3, 2)}, {(1, 1), (2, 1), (4, 2), (5, 3)}),
    ("कृपया दरवाजा बंद करें।", "దయచేసి తలుపు మూయండి।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),
    ("वह गाना बहुत मधुर है।", "ఆ పాట చాలా మధురంగా ఉంది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 5)}),
    ("सफलता के लिए कड़ी मेहनत जरूरी है।", "విజయానికి కష్టపడటం అవసరం।", {(0, 0), (4, 1), (5, 2)}, {(1, 0), (2, 0), (3, 1), (6, 2), (7, 3)}),
    ("हम सब भारतीय हैं।", "మనమందరం భారతీయులం।", {(0, 0), (1, 0), (2, 1)}, {(3, 1), (4, 2)}),

    # Batch 2: Nature, Food, Travel
    ("बगीचे में फूल खिल रहे हैं।", "తోటలో పూలు పూస్తున్నాయి।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),
    ("नदी घाटी के बीच से बहती है।", "నది లోయ గుండా ప్రవహిస్తుంది।", {(0, 0), (1, 1), (4, 3)}, {(2, 1), (3, 1), (5, 4)}),
    ("मुझे गर्मियों में आम खाना पसंद है।", "నాకు వేసవిలో మామిడి పండ్లు తినడం ఇష్టం।", {(0, 0), (3, 2), (4, 3), (5, 4)}, {(1, 1), (2, 1), (6, 5)}),
    ("वह हमारे लिए स्वादिष्ट खाना बना रही है।", "ఆమె మా కోసం రుచికరమైన ఆహారాన్ని వండుతోంది।", {(0, 0), (2, 2), (3, 3), (4, 4)}, {(1, 1), (5, 4), (6, 5)}),
    ("आज ट्रेन देर से आ रही है।", "రైలు ఈ రోజు ఆలస్యంగా ఉంది।", {(0, 1), (1, 0), (2, 2)}, {(3, 3), (4, 4), (5, 4)}),
    ("मैं ताज महल देखना चाहता हूँ।", "నేను తాజ్ మహల్ దర్శించాలనుకుంటున్నాను।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 3)}),
    ("खाना खाने के बाद यह दवा लें।", "భోజనం తర్వాత ఈ మందు తీసుకోండి।", {(0, 3), (1, 0), (2, 1), (4, 2), (5, 4)}, {(3, 1), (6, 4)}),
    ("डॉक्टर ने उसे आराम करने की सलाह दी।", "వైద్యుడు అతనికి విశ్రాంతి తీసుకోమని సలహా ఇచ్చాడు।", {(0, 0), (2, 1), (3, 2), (6, 4)}, {(1, 0), (4, 2), (5, 3), (7, 5)}),
    ("मोबाइल फोन ने हमारी जिंदगी बदल दी है।", "మొబైల్ ఫోన్లు మన జీవితాలను మార్చాయి।", {(0, 0), (1, 1), (3, 3), (4, 4)}, {(2, 2), (5, 5), (6, 5)}),
    ("इंटरनेट ज्ञान का एक बड़ा स्रोत है।", "ఇంటర్నెట్ జ్ఞానానికి ఒక మూలం।", {(0, 0), (1, 1), (5, 4)}, {(2, 1), (3, 2), (4, 3), (6, 5)}),

    # Batch 3: More variations
    ("किताबें पढ़ना अच्छी आदत है।", "పుస్తకాలు చదవడం మంచి అలవాటు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("क्या आप अंग्रेजी बोल सकते हैं?", "మీరు ఇంగ్లీష్ మాట్లాడగలరా?", {(1, 0), (2, 1), (3, 2)}, {(0, 2), (4, 2), (5, 3)}),
    ("हिमालय दुनिया का सबसे ऊँचा पहाड़ है।", "హిమాలయాలు ప్రపంచంలోనే ఎత్తైన పర్వతం।", {(0, 0), (1, 1), (4, 3), (5, 4)}, {(2, 1), (3, 2), (6, 5)}),
    ("पेड़ हमें ऑक्सीजन देते हैं।", "చెట్లు మనకు ఆక్సిజన్ ఇస్తాయి।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("समय बहुत कीमती होता है।", "సమయం చాలా విలువైనది।", {(0, 0), (1, 1), (2, 2)}, {(3, 3), (4, 3)}),
    ("मेहनत कभी बेकार नहीं जाती।", "కష్టపడటం ఎప్పుడూ వృథా పోదు।", {(0, 0), (3, 3), (4, 4)}, {(1, 1), (2, 2), (5, 4)}),
    ("हम कल फिल्म देखने जाएंगे।", "మనం రేపు సినిమా చూడటానికి వెళ్తాము।", {(0, 0), (1, 1), (2, 2), (4, 4)}, {(3, 3), (5, 5)}),
    ("आज बहुत ठंड है।", "ఈ రోజు చాలా చలిగా ఉంది।", {(0, 0), (1, 1), (2, 3)}, {(3, 4)}),
    ("वह बहुत सुंदर चित्र बनाती है।", "ఆమె చాలా అందమైన చిత్రాలను గీస్తుంది।", {(0, 0), (2, 2), (3, 3), (4, 4)}, {(1, 1), (5, 4)}),
    ("सत्य की हमेशा जीत होती है।", "నిజం ఎప్పుడూ గెలుస్తుంది।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),

    # Batch 4: Everyday life and objects
    ("मेरे पास एक काला कुत्ता है।", "నా దగ్గర ఒక నల్ల కుక్క ఉంది।", {(0, 0), (1, 1), (3, 3), (4, 4)}, {(2, 2), (5, 5)}),
    ("दूध सेहत के लिए अच्छा है।", "పాలు ఆరోగ్యానికి మంచిది।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3), (5, 3)}),
    ("चिड़िया आसमान में उड़ रही है।", "పక్షి ఆకాశంలో ఎగురుతోంది।", {(0, 0), (1, 1), (2, 1), (3, 3)}, {(4, 3), (5, 4)}),
    ("यह घर बहुत बड़ा और सुंदर है।", "ఈ ఇల్లు చాలా పెద్దది మరియు అందంగా ఉంది।", {(0, 0), (1, 1), (3, 3), (4, 4), (5, 5)}, {(2, 2), (6, 6)}),
    ("मुझे नीले रंग की कमीज चाहिए।", "నాకు నీలం రంగు చొక్కా కావాలి।", {(0, 0), (1, 1), (2, 2), (4, 3), (5, 4)}, {(3, 2), (6, 5)}),
    ("वह रोज सुबह दौड़ने जाता है।", "అతను ప్రతిరోజూ ఉదయం పరుగెత్తడానికి వెళ్తాడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 4), (5, 4)}),
    ("यहाँ धूम्रपान मना है।", "ఇక్కడ ధూమపానం నిషేధించబడింది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("गाड़ी बहुत तेज चल रही है।", "బండి చాలా వేగంగా వెళ్తోంది।", {(0, 0), (2, 2), (3, 3)}, {(1, 1), (4, 3), (5, 4)}),
    ("मैं संगीत सुनना पसंद करता हूँ।", "నేను సంగీతం వినడం ఇష్టపడతాను।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("मेरे पिता एक किसान हैं।", "మా తండ్రి ఒక రైతు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3)}),

    # Batch 5: Abstract and variations
    ("अंधेरे के बाद रोशनी आती है।", "చీకటి తర్వాత వెలుగు వస్తుంది।", {(0, 0), (1, 1), (3, 2), (4, 3)}, {(2, 1), (5, 3)}),
    ("क्षमा करना एक महान गुण है।", "క్షమించడం ఒక గొప్ప గుణం।", {(0, 0), (2, 1), (3, 2), (4, 3)}, {(1, 1), (5, 4)}),
    ("साहस से डरो मत।", "ధైర్యంతో భయపడవద్దు।", {(0, 0), (2, 1)}, {(1, 0), (3, 2)}),
    ("दोस्ती दुनिया का सबसे अनमोल रिश्ता है।", "స్నేహం ప్రపంచంలోనే అతి విలువైన బంధం।", {(0, 0), (1, 1), (4, 3), (5, 4)}, {(2, 1), (3, 2), (6, 5)}),
    ("आज रात चाँद चमक रहा है।", "ఈ రాత్రి చంద్రుడు ప్రకాశిస్తున్నాడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3), (5, 4)}),
    ("वह हमेशा सच बोलता है।", "అతను ఎప్పుడూ నిజం చెబుతాడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("जीवन एक संघर्ष है।", "జీవితం ఒక పోరాటం।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("हमें बड़ों का सम्मान करना चाहिए।", "మనం పెద్దలను గౌరవించాలి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2), (5, 3)}),
    ("ज्ञान सबसे शक्तिशाली हथियार है।", "జ్ఞానం అత్యంత శక్తివంతమైన ఆయుధం।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3)}),
    ("अपनी मेहनत पर भरोसा रखें।", "నీ కష్టాన్ని నమ్ముకో।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2)}),

    # Batch 6: More Travel and Places
    ("स्टेशन यहाँ से दूर है।", "స్టేషన్ ఇక్కడి నుండి దూరంగా ఉంది।", {(0, 0), (1, 1), (3, 3)}, {(2, 2), (4, 3), (5, 4)}),
    ("बस स्टैंड कहाँ है?", "బస్టాండ్ ఎక్కడ ఉంది?", {(0, 0), (1, 0), (2, 1)}, {(3, 2)}),
    ("मुझे टिकट खरीदना है।", "నేను టికెట్ కొనాలి।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("अपनी सीट पर बैठ जाइए।", "మీ సీటులో కూర్చోండి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2)}),
    ("हवाई जहाज आसमान में उड़ रहा है।", "విమానం ఆకాశంలో ఎగురుతోంది।", {(0, 0), (1, 0), (2, 1), (4, 2)}, {(3, 1), (5, 2), (6, 3)}),
    ("रास्ता बहुत ऊबड़-खाबड़ है।", "దారి చాలా గతుకులుగా ఉంది।", {(0, 0), (1, 1), (2, 2)}, {(3, 3), (4, 3)}),
    ("पहाड़ों की सुंदरता अद्भुत है।", "పర్వతాల అందం అద్భుతంగా ఉంది।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),
    ("समुद्र की लहरें ऊँची हैं।", "సముద్రపు అలలు ఎత్తుగా ఉన్నాయి।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 3), (5, 3)}),
    ("शहर में बहुत भीड़ है।", "నగరంలో చాలా రద్దీగా ఉంది।", {(0, 0), (2, 2)}, {(1, 0), (3, 2), (4, 3)}),
    ("गाँव की शांति पसंद है।", "గ్రామపు ప్రశాంతత ఇష్టం।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 2)}),

    # Batch 7: Food and Health
    ("मुझे प्यास लगी है।", "నాకు దాహంగా ఉంది।", {(0, 0), (1, 1)}, {(2, 2), (3, 2)}),
    ("ताजा फल खाओ।", "తాజా పండ్లు తిను।", {(0, 0), (1, 1), (2, 2)}, {}),
    ("सब्जियां सेहत के लिए अच्छी होती हैं।", "కూరగాయలు ఆరోగ్యానికి మంచివి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3), (5, 3)}),
    ("पानी उबाल कर पिएं।", "నీటిని మరిగించి తాగండి।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 2)}),
    ("योगा करने से मन शांत रहता है।", "యోగా చేయడం వల్ల మనస్సు ప్రశాంతంగా ఉంటుంది।", {(0, 0), (1, 1), (3, 3), (4, 4)}, {(2, 2), (5, 5), (6, 5)}),
    ("हाथ धोकर खाना खाएं।", "చేతులు కడుక్కుని భోజనం చేయండి।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("ज्यादा चीनी मत खाओ।", "ఎక్కువ చక్కెర తినవద్దు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {}),
    ("नींद पूरी होनी चाहिए।", "నిద్ర పూర్తి కావాలి।", {(0, 0), (1, 1)}, {(2, 2), (3, 2)}),
    ("स्वच्छता बहुत जरूरी है।", "పరిశుభ్రత చాలా అవసరం।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("वह अस्पताल जा रहा है।", "అతను ఆసుపత్రికి వెళ్తున్నాడు।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),

    # Batch 8: Education and Work
    ("स्कूल समय पर जाओ।", "పాఠశాలకు సమయానికి వెళ్ళు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1)}),
    ("शिक्षक पाठ पढ़ा रहे हैं।", "ఉపాధ్యాయుడు పాఠం చెబుతున్నారు।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),
    ("परीक्षा की तैयारी करो।", "పరీక్షకు సిద్ధమవ్వండి।", {(0, 0), (2, 1)}, {(1, 0), (3, 1)}),
    ("सवाल का जवाब दो।", "ప్రశ్నకు సమాధానం ఇవ్వండి।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 2)}),
    ("लिखना और पढ़ना सीखो।", "రాయడం మరియు చదవడం నేర్చుకోండి।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("वह दफ्तर में काम करता है।", "అతను కార్యాలయంలో పని చేస్తాడు।", {(0, 0), (1, 1), (3, 2)}, {(2, 1), (4, 3), (5, 4)}),
    ("ईमेल भेज दो।", "ఈమెయిల్ పంపండి।", {(0, 0), (1, 1), (2, 1)}, {}),
    ("मीटिंग कल होगी।", "సమావేశం రేపు జరుగుతుంది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("मेरा कंप्यूटर खराब है।", "నా కంప్యూటర్ పాడైపోయింది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2)}),
    ("नई चीजें सीखना अच्छा है।", "కొత్త విషయాలు నేర్చుకోవడం మంచిది।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),

    # Batch 9: Relationships and Feelings
    ("मेरी माँ बहुत प्यार करती है।", "మా అమ్మ చాలా ప్రేమిస్తుంది।", {(0, 0), (1, 1), (3, 3)}, {(2, 2), (4, 3)}),
    ("भाई-बहन साथ खेलते हैं।", "అన్నదమ్ములు అక్కాచెల్లెళ్లు కలిసి ఆడుకుంటారు।", {(0, 0), (2, 1), (3, 2)}, {(1, 0), (4, 2)}),
    ("वह मेरा सच्चा मित्र है।", "అతను నా నిజమైన స్నేహితుడు।", {(0, 0), (1, 1), (2, 2), (3, 3)}, {(4, 3)}),
    ("बच्चे शोर मचा रहे हैं।", "పిల్లలు అల్లరి చేస్తున్నారు।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 3)}),
    ("मुझे गुस्सा आ रहा है।", "నాకు కోపం వస్తోంది।", {(0, 0), (1, 1), (2, 2)}, {(3, 2), (4, 2)}),
    ("डरने की कोई बात नहीं।", "భయపడాల్సిన అవసరం లేదు।", {(0, 0), (3, 2)}, {(1, 0), (2, 1), (4, 2)}),
    ("खुश रहना सबसे जरूरी है।", "సంతోషంగా ఉండటం అత్యంత ముఖ్యం।", {(0, 0), (1, 1), (3, 3)}, {(2, 2), (4, 3)}),
    ("वह उदास क्यों है?", "అతను ఎందుకు విచారంగా ఉన్నాడు?", {(0, 0), (1, 1), (2, 2)}, {(3, 3), (4, 3)}),
    ("सबका भला करो।", "అందరికీ మంచి చేయి।", {(0, 0), (1, 1), (2, 2)}, {}),
    ("हम सब एक हैं।", "మనమందరం ఒక్కటే।", {(0, 0), (1, 0), (2, 1)}, {(3, 1)})
]

# --- EXECUTION WITH PASS/FAIL LOGIC ---

def run_pipeline(quality_threshold=0.35):
    aligner = UnsupervisedAligner()
    passed = 0
    failed = 0

    print(f"\n{'ID':<3} | {'QSCORE':<7} | {'STATUS':<6} | {'MATCHES'}")
    print("-" * 70)

    for i, (hi_text, te_text, _, _) in enumerate(HINDI_TELUGU_DATASET):
        links, q_score = aligner.align_and_evaluate(hi_text, te_text)

        # Pass/Fail Logic
        status = "✅ PASS" if q_score >= quality_threshold else "❌ FAIL"
        if q_score >= quality_threshold: passed += 1
        else: failed += 1

        top_3 = " | ".join([f"{l[0]}↔{l[1]}" for l in sorted(links, key=lambda x: x[2], reverse=True)[:2]])
        print(f"{i+1:<3} | {q_score:.4f} | {status:<6} | {top_3}...")

    # --- FINAL REPORT ---
    print("\n" + "="*30)
    print(f"📊 FINAL ALIGNMENT REPORT")
    print(f"Total Sentences: {len(HINDI_TELUGU_DATASET)}")
    print(f"Passed (High Quality): {passed}")
    print(f"Failed (Low Confidence): {failed}")
    print(f"Success Rate: {(passed/len(HINDI_TELUGU_DATASET))*100:.2%}")
    print("="*30)

if __name__ == "__main__":
    run_pipeline()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



ID  | QSCORE  | STATUS | MATCHES
----------------------------------------------------------------------
1   | 0.7750 | ✅ PASS | मैं↔నేను | स्कूल↔పాఠశాలకు...
2   | 0.9000 | ✅ PASS | आज↔రోజు | मौसम↔వాతావరణం...
3   | 0.7857 | ✅ PASS | आप↔మీరు | मेरी↔నాకు...
4   | 0.8000 | ✅ PASS | मुझे↔నాకు | भूख↔ఆకలిగా...
5   | 0.8333 | ✅ PASS | वह↔అతను | मेरा↔నా...
6   | 0.9000 | ✅ PASS | भारत↔భారతదేశం | एक↔ఒక...
7   | 0.8333 | ✅ PASS | कल↔రేపు | छुट्टी↔సెలవు...
8   | 0.9000 | ✅ PASS | आपको↔మిమ్మల్ని | यहाँ↔ఇక్కడ...
9   | 0.7500 | ✅ PASS | पानी↔నీరు | जीवन↔జీవించడానికి...
10  | 0.7500 | ✅ PASS | सूर्य↔సూర్యుడు | पूर्व↔తూర్పున...
11  | 1.0000 | ✅ PASS | मैं↔నేను | आपसे↔మిమ్మల్ని...
12  | 1.0000 | ✅ PASS | यह↔ఈ | फिल्म↔సినిమా...
13  | 0.9000 | ✅ PASS | मुझे↔నాకు | कॉफी↔కాఫీ...
14  | 0.7500 | ✅ PASS | बच्चे↔పిల్లలు | बगीचे↔తోటలో...
15  | 0.8000 | ✅ PASS | दिल्ली↔ఢిల్లీ | भारत↔భారతదేశ...
16  | 0.8000 | ✅ PASS | ईमानदारी↔నిజాయితీ | अच्छी↔ఉత్తమ...
17  | 0.8750 | ✅ PASS | कृपया↔దయచేసి | दरवाजा↔తలుపు...
18  | 

In [ ]:
import torch
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from torch.nn.functional import softmax

class DeepAligner:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = SentenceTransformer('sentence-transformers/LaBSE').to(self.device)
        self.tokenizer = self.model.tokenizer

    def _get_vectors(self, tokens):
        inputs = self.tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.model[0].auto_model(**inputs, output_hidden_states=True)
            # Using Layer 8 for deep semantic features
            embeddings = outputs.hidden_states[8][0]

        word_ids = inputs.word_ids(batch_index=0)
        return torch.stack([embeddings[[j for j, w_id in enumerate(word_ids) if w_id == i]].mean(dim=0)
                            for i in range(len(tokens))])

    def align_complex(self, hi_text, te_text):
        hi_toks = re.sub(r'[।\?\!\,\(\)"]', '', hi_text).strip().split()
        te_toks = re.sub(r'[।\?\!\,\(\)"]', '', te_text).strip().split()

        hi_vecs = self._get_vectors(hi_toks)
        te_vecs = self._get_vectors(te_toks)

        # Compute Double Softmax with High Temperature for Precision
        sim_mat = torch.matmul(hi_vecs, te_vecs.T)
        mutual_conf = (softmax(sim_mat * 12.0, dim=1) * softmax(sim_mat * 12.0, dim=0)).cpu().numpy()

        links = []
        for i in range(len(hi_toks)):
            for j in range(len(te_toks)):
                if mutual_conf[i, j] > 0.01: # Lower threshold to see ALL connections
                    links.append((hi_toks[i], te_toks[j], mutual_conf[i, j]))

        avg_conf = np.mean([l[2] for l in links]) if links else 0
        coverage = len(links) / max(len(hi_toks), len(te_toks))
        return links, (avg_conf + coverage) / 2

# --- LONG & COMPLEX DATASET SAMPLES (Expand to 100) ---
# 100 COMPLEX HINDI-TELUGU SENTENCE PAIRS
# Format: (Hindi_Sentence, Telugu_Sentence)

COMPLEX_DATASET = [
    # --- I. LEGAL & GOVERNANCE ---
    ("भारतीय संविधान की प्रस्तावना में निहित धर्मनिरपेक्षता, समाजवाद और अखंडता के आदर्श राष्ट्र की एकता को सुदृढ़ करते हैं।", "భారత రాజ్యాంగ పీఠికలో పొందుపరచబడిన లౌకికవాదం, సామ్యవాదం మరియు అఖండత అనే ఆదర్శాలు దేశ ఐక్యతను బలోపేతం చేస్తాయి।"),
    ("उच्चतम न्यायालय ने निर्णय दिया कि निजता का अधिकार संविधान के अनुच्छेद 21 के तहत जीवन और व्यक्तिगत स्वतंत्रता का अभिन्न अंग है।", "గోప్యత హక్కు రాజ్యాంగంలోని 21వ అధికరణం ప్రకారం జీవించే మరియు వ్యక్తిగత స్వేచ్ఛలో అంతర్భాగమని సుప్రీంకోర్టు తీర్పునిచ్చింది।"),
    ("द्विसदनीय विधायिका की संरचना यह सुनिश्चित करती है कि केंद्र और राज्यों के बीच शक्तियों का वितरण संतुलन में बना रहे।", "ద్విసభా శాసనసభ నిర్మాణం కేంద్రం మరియు రాష్ట్రాల మధ్య అధికారాల పంపిణీ సమతుల్యంగా ఉండేలా చూస్తుంది।"),
    ("चुनाव आयोग एक स्वायत्त संवैधानिक प्राधिकरण है जो भारत में संघ और राज्य चुनाव प्रक्रियाओं के प्रशासन के लिए उत्तरदायी है।", "ఎన్నికల సంఘం అనేది భారతదేశంలో కేంద్ర మరియు రాష్ట్ర ఎన్నికల ప్రక్రియల నిర్వహణకు బాధ్యత వహించే ఒక స్వయంప్రతిపత్తి కలిగిన రాజ్యాంగ సంస్థ।"),
    ("लोकपाल और लोकायुक्त अधिनियम का उद्देश्य सार्वजनिक पदाधिकारियों के विरुद्ध भ्रष्टाचार के आरोपों की निष्पक्ष जांच करना है।", "లోక్‌పాల్ మరియు లోకాయుక్త చట్టం యొక్క ఉద్దేశ్యం ప్రభుత్వ అధికారులపై వచ్చే అవినీతి ఆరోపణలపై నిష్పక్షపాతంగా విచారణ జరపడం।"),
    ("मौलिक कर्तव्यों का पालन करना प्रत्येक नागरिक का नैतिक उत्तरदायित्व है ताकि समाज में सद्भाव और अनुशासन बना रहे।", "సమాజంలో సామరస్యం మరియు క్రమశిక్షణను కాపాడటానికి ప్రతి పౌరుడు ప్రాథమిక విధులను పాటించడం నైతిక బాధ్యత।"),
    ("संसद के दोनों सदनों द्वारा पारित विधेयक को राष्ट्रपति की सहमति प्राप्त होने के बाद ही आधिकारिक कानून माना जाता है।", "పార్లమెంటు ఉభయ సభలు ఆమోదించిన బిల్లుకు రాష్ట్రపతి అంగీకారం లభించిన తర్వాతే అది అధికారిక చట్టంగా మారుతుంది।"),
    ("प्रशासनिक विकेंद्रीकरण का मुख्य उद्देश्य स्थानीय स्वशासन संस्थानों को सशक्त बनाना और जमीनी स्तर पर विकास सुनिश्चित करना है।", "పరిపాలనా వికేంద్రీకరణ యొక్క ప్రధాన ఉద్దేశ్యం స్థానిక స్వపరిపాలన సంస్థలను బలోపేతం చేయడం మరియు క్షేత్రస్థాయిలో అభివృద్ధిని నిర్ధారించడం।"),
    ("न्यायिक सक्रियता के माध्यम से अदालतें अक्सर कार्यपालिका को उनके संवैधानिक कर्तव्यों के प्रति सचेत करने का कार्य करती हैं।", "న్యాయవ్యవస్థ క్రియాశీలత ద్వారా కోర్టులు తరచుగా కార్యనిర్వాహక వ్యవస్థను వారి రాజ్యాంగ విధులను గుర్తు చేస్తూ ఉంటాయి।"),
    ("अन्तरराष्ट्रीय न्यायालय का मुख्य कार्य राज्यों के बीच कानूनी विवादों को सुलझाना और कानूनी प्रश्नों पर सलाह देना है।", "అంతర్జాతీయ న్యాయస్థానం యొక్క ప్రధాన విధి దేశాల మధ్య చట్టపరమైన వివాదాలను పరిష్కరించడం మరియు చట్టపరమైన ప్రశ్నలపై సలహాలు ఇవ్వడం।"),

    # --- II. SPACE & PHYSICS ---
    ("भारतीय अंतरिक्ष अनुसंधान संगठन ने मंगलयान मिशन के सफल प्रक्षेपण के साथ अंतरग्रहीय अन्वेषण में नया अध्याय लिखा।", "భారత అంతరిక్ష పరిశోధనా సంస్థ మంగళయాన్ మిషన్ విజయవంత ప్రయోగంతో అంతర గ్రహ పరిశోధనలో కొత్త అధ్యాయాన్ని లిఖించింది।"),
    ("क्वांटम भौतिकी के सिद्धांत बताते हैं कि सूक्ष्म स्तर पर कणों का व्यवहार शास्त्रीय भौतिकी के नियमों से भिन्न होता है।", "క్వాంటం భౌతికశాస్త్ర సూత్రాలు సూక్ష్మ స్థాయిలో కణాల ప్రవర్తన శాస్త్రీయ భౌతికశాస్త్ర నియమాల కంటే భిన్నంగా ఉంటుందని చెబుతాయి।"),
    ("पारिस्थितिक तंत्र में जैव विविधता की हानि खाद्य श्रृंखला के स्थायित्व को गंभीर रूप से असंतुलित कर सकती है।", "పర్యావరణ వ్యవస్థలో జీవవైవిధ్యం తగ్గడం వల్ల ఆహార గొలుసు యొక్క స్థిరత్వం తీవ్రంగా దెబ్బతింటుంది।"),
    ("वैश्विक तापन के कारण ग्लेशियरों का पिघलना समुद्र के स्तर में वृद्धि और तटीय क्षेत्रों के जलमग्न होने का कारण है।", "భూతాపం వల్ల హిమానీనదాలు కరగడం సముద్ర మట్టాలు పెరగడానికి మరియు తీర ప్రాంతాలు మునిగిపోవడానికి కారణం।"),
    ("प्रकाश की दोहरी प्रकृति यह सिद्ध करती है कि वह तरंग और कण दोनों के रूप में व्यवहार कर सकता है।", "కాంతి యొక్క ద్వంద్వ స్వభావం అది తరంగం మరియు కణం రెండింటి రూపంలో ప్రవర్తించగలదని నిరూపిస్తుంది।"),
    ("ब्लैक होल अंतरिक्ष में वह स्थान है जहाँ गुरुत्वाकर्षण इतना प्रबल होता है कि प्रकाश भी वहां से नहीं निकल सकता।", "బ్లాక్ హోల్ అనేది అంతరిక్షంలో గురుత్వాకర్షణ శక్తి ఎంత బలంగా ఉంటుందంటే అక్కడ నుండి కాంతి కూడా తప్పించుకోలేదు।"),
    ("सौरमंडल के ग्रहों की गति को समझने के लिए केप्लर के नियमों का अध्ययन करना खगोल विज्ञान में अनिवार्य है।", "సౌర వ్యవస్థలోని గ్రహాల గమనాన్ని అర్థం చేసుకోవడానికి కెప్లర్ నియమాలను అధ్యయనం చేయడం ఖగోళ శాస్త్రంలో తప్పనిసరి।"),
    ("परमाणु रिएक्टरों में नियंत्रित नाभिकीय विखंडन की प्रक्रिया के माध्यम से भारी मात्रा में ऊर्जा उत्पन्न की जाती है।", "అణు రియాక్టర్లలో నియంత్రిత కేంద్రక విచ్ఛిత్తి ప్రక్రియ ద్వారా భారీ మొత్తంలో ఇంధనం ఉత్పత్తి చేయబడుతుంది।"),
    ("ओजोन परत वायुमंडल के ऊपरी भाग में स्थित है जो हानिकारक पराबैंगनी विकिरण को पृथ्वी तक पहुँचने से रोकती है।", "ఓజోన్ పొర వాతావరణం పైభాగంలో ఉండి హానికరమైన అతినీలలోహిత వికిరణం భూమికి చేరకుండా అడ్డుకుంటుంది।"),
    ("आइंस्टीन का सापेक्षता का सिद्धांत स्थान और समय के बीच के संबंध को वैज्ञानिक रूप से पुनर्परिभाषित करता है।", "ఐన్‌స్టీన్ సాపేక్షత సిద్ధాంతం స్థలం మరియు కాలం మధ్య సంబంధాన్ని శాస్త్రీయంగా పునర్నిర్వచించింది।"),

    # --- III. TECHNOLOGY & AI ---
    ("कृत्रिम बुद्धिमत्ता के एल्गोरिदम डेटा के विशाल भंडार का विश्लेषण करके भविष्य के रुझानों की सटीक भविष्यवाणी करते हैं।", "కృత్రిమ మేధస్సు అల్గారిథమ్స్ భారీ డేటాను విశ్లేషించి భవిష్యత్తు ధోరణులను ఖచ్చితంగా అంచనా వేస్తాయి।"),
    ("ब्लॉकचेन तकनीक विकेंद्रीकृत बहीखाता प्रणाली के माध्यम से वित्तीय लेनदेन की सुरक्षा और पारदर्शिता सुनिश्चित करती है।", "బ్లాక్‌చైన్ సాంకేతికత వికేంద్రీకృత లెడ్జర్ వ్యవస్థ ద్వారా ఆర్థిక లావాదేవీల భద్రత మరియు పారదర్శకతను నిర్ధారిస్తుంది।"),
    ("वस्तु एवं सेवा कर (GST) के कार्यान्वयन ने भारत में अप्रत्यक्ष कर ढांचे को एकीकृत करके साझा बाजार बनाया है।", "వస్తు సేవల పన్ను (GST) అమలు భారతదేశంలో పరోక్ష పన్ను నిర్మాణాన్ని ఏకీకృతం చేయడం ద్వారా ఉమ్మడి మార్కెట్‌ను సృష్టించింది।"),
    ("सूक्ष्म, लघु और मध्यम उद्यम क्षेत्र भारतीय अर्थव्यवस्था की रीढ़ है जो ग्रामीण रोजगार सृजन में योगदान देता है।", "సూక్ష్మ, చిన్న మరియు మధ్య తరహా పరిశ్రమల రంగం భారత ఆర్థిక వ్యవస్థకు వెన్నెముక, ఇది గ్రామీణ ఉపాధి కల్పనలో తోడ్పడుతుంది।"),
    ("साइबर सुरक्षा की चुनौतियाँ निरंतर बढ़ती जा रही हैं क्योंकि डिजिटल बुनियादी ढाँचा अधिक जटिल होता जा रहा है।", "డిజిటల్ మౌలిక సదుపాయాలు మరింత సంక్లిష్టంగా మారుతున్న కొద్దీ సైబర్ భద్రతా సవాళ్లు నిరంతరం పెరుగుతున్నాయి।"),
    ("मशीन लर्निंग के मॉडल जितना अधिक डेटा प्राप्त करते हैं, उनकी सटीकता और प्रदर्शन में उतना ही सुधार होता है।", "మెషిన్ లెర్నింగ్ మోడల్స్ ఎంత ఎక్కువ డేటాను పొందితే, వాటి ఖచ్చితత్వం మరియు పనితీరు అంత మెరుగుపడుతుంది।"),
    ("क्लाउड कंप्यूटिंग व्यवसायों को भौतिक बुनियादी ढांचे में निवेश किए बिना बड़े पैमाने पर डेटा स्टोर करने की अनुमति देती है।", "క్లౌడ్ కంప్యూటింగ్ భౌతిక మౌలిక సదుపాయాలపై పెట్టుబడి పెట్టకుండానే భారీ డేటాను భద్రపరిచేందుకు వ్యాపారాలకు అనుమతిస్తుంది।"),
    ("इंटरनेट ऑफ थिंग्स (IoT) विभिन्न उपकरणों को एक नेटवर्क से जोड़कर स्मार्ट स्वचालन की सुविधा प्रदान करता है।", "ఇంటర్నెట్ ఆఫ్ థింగ్స్ (IoT) వివిధ పరికరాలను నెట్‌వర్క్‌తో అనుసంధానించడం ద్వారా స్మార్ట్ ఆటోమేషన్‌కు వీలు కల్పిస్తుంది।"),
    ("स्वदेशी विनिर्माण को बढ़ावा देने के लिए 'मेक इन इंडिया' पहल ने रक्षा उत्पादन क्षेत्र में निवेश को आकर्षित किया है।", "స్వదేశీ తయారీని ప్రోత్సహించడానికి 'మేక్ ఇన్ ఇండియా' చొరవ రక్షణ ఉత్పత్తి రంగంలో పెట్టుబడులను ఆకర్షించింది।"),
    ("आर्थिक मंदी के दौरान सरकारें अक्सर तरलता बढ़ाने के लिए ब्याज दरों को कम करने जैसे राजकोषीय उपाय करती हैं।", "ఆర్థిక మాంద్యం సమయంలో ప్రభుత్వం లిక్విడిటీ పెంచడానికి వడ్డీ రేట్లను తగ్గించడం వంటి చర్యలు తీసుకుంటుంది।"),

    # --- IV. PHILOSOPHY & HISTORY ---
    ("अद्वैत वेदांत दर्शन के अनुसार आत्मा और परमात्मा के बीच का भेद केवल अविद्या या अज्ञान के कारण प्रतीत होता है।", "అద్వైత వేదాంత దర్శనం ప్రకారం ఆత్మ మరియు పరమాత్మ మధ్య భేదం కేవలం అవిద్య లేదా అజ్ఞానం వల్లనే కనిపిస్తుంది।"),
    ("मौर्य साम्राज्य के शिलालेख राजा अशोक की धम्म नीति और लोक कल्याणकारी प्रशासन के जीवंत प्रमाण प्रस्तुत करते हैं।", "మౌర్య సామ్రాజ్య శిలాశాసనాలు అశోక చక్రవర్తి ధమ్మ నీతి మరియు ప్రజా సంక్షేమ పాలనకు సజీవ సాక్ష్యాలుగా నిలుస్తాయి।"),
    ("सांस्कृतिक बहुलवाद भारतीय समाज की वह विशेषता है जो विभिन्न भाषाओं और परंपराओं को एक सूत्र में पिरोती है।", "సాంస్కృతిక బహుళత్వం అనేది భారతీయ సమాజం యొక్క ప్రత్యేకత, ఇది వివిధ భాషలను మరియు సంప్రదాయాలను ఒకే త్రాడుపైకి తెస్తుంది।"),
    ("पुनर्जागरण काल ने कला और विज्ञान के क्षेत्र में तर्क और मानवतावाद को धार्मिक कट्टरता पर वरीयता दी।", "పునర్జీవన కాలం కళ మరియు విజ్ఞాన రంగాలలో మతపరమైన ఛాందసవాదం కంటే తర్కానికి మరియు మానవతావాదానికి ప్రాధాన్యతనిచ్చింది।"),
    ("प्राचीन भारतीय चिकित्सा पद्धति आयुर्वेद रोगों के उपचार के बजाय जीवनशैली के संतुलन पर अधिक बल देती है।", "ప్రాచీన భారతీయ వైద్య విధానం ఆయుర్వేదం వ్యాధుల చికిత్స కంటే జీవనశైలి సమతుల్యతపై ఎక్కువ మొగ్గు చూపుతుంది।"),
    ("बुद्ध के अष्टांगिक मार्ग का उद्देश्य मानवीय दुखों का अंत करना और निर्वाण की प्राप्ति के लिए मार्गदर्शन करना है।", "బుద్ధుని అష్టాంగ మార్గం మానవ దుఃఖాలను అంతం చేయడానికి మరియు నిర్వాణం సాధించడానికి మార్గదర్శకత్వం చేస్తుంది।"),
    ("सिंधु घाटी सभ्यता की शहरी योजना और जल निकासी प्रणाली उस समय की उन्नत तकनीकी समझ को दर्शाती है।", "సింధు లోయ నాగరికత యొక్క పట్టణ ప్రణాళిక మరియు డ్రైనేజీ వ్యవస్థ అప్పటి అధునాతన సాంకేతిక అవగాహనను ప్రతిబింబిస్తాయి।"),
    ("भक्ति आंदोलन ने जातिगत बंधनों को तोड़कर ईश्वर के प्रति व्यक्तिगत समर्पण और प्रेम के मार्ग को लोकप्रिय बनाया।", "భక్తి ఉద్యమం కుల బంధాలను తెంచి భగవంతుని పట్ల వ్యక్తిగత అంకితభావం మరియు ప్రేమ మార్గాన్ని ప్రాచుర్యంలోకి తెచ్చింది।"),
    ("गांधीवादी दर्शन में सत्य और अहिंसा को केवल आदर्श नहीं बल्कि सामाजिक परिवर्तन के अस्त्र के रूप में देखा गया है।", "గాంధేయ దర్శనంలో సత్యం మరియు అహింస కేవలం ఆదర్శాలు మాత్రమే కాదు, సామాజిక మార్పుకు అస్త్రాలుగా చూడబడ్డాయి।"),
    ("मुगल वास्तुकला में भारतीय और फारसी शैलियों का अद्भुत समन्वय ताजमहल जैसी भव्य इमारतों में परिलक्षित होता है।", "మొఘల్ వాస్తుశిల్పంలో భారతీయ మరియు పర్షియన్ శైలుల అద్భుత సమన్వయం తాజ్ మహల్ వంటి భవనాలలో కనిపిస్తుంది।"),

    # --- V. EDUCATION & ETHICS (41-60) ---
    ("नई शिक्षा नीति का उद्देश्य छात्रों में रटने की प्रवृत्ति के बजाय आलोचनात्मक सोच और रचनात्मकता विकसित करना है।", "కొత్త విద్యా విధానం విద్యార్థులలో బట్టీ పట్టే ధోరణి కంటే విమర్శనాత్మక ఆలోచన మరియు సృజనాత్మకతను పెంపొందించడం లక్ష్యంగా పెట్టుకుంది।"),
    ("समावेशी विकास का अर्थ है कि आर्थिक प्रगति का लाभ समाज के सबसे अंतिम पायदान पर खड़े व्यक्ति तक पहुँचे।", "సమ్మిళిత వృద్ధి అంటే ఆర్థిక పురోగతి యొక్క ప్రయోజనాలు సమాజంలోని చివరి వ్యక్తికి కూడా అందడం।"),
    ("नैतिक मूल्य ही वह आधार हैं जिस पर एक स्वस्थ और न्यायपूर्ण समाज की नींव रखी जा सकती है।", "నైతిక విలువల ప్రాతిపదికన మాత్రమే ఆరోగ్యకరమైన మరియు న్యాయమైన సమాజ పునాది నిర్మించబడుతుంది।"),
    ("तकनीकी शिक्षा के साथ-साथ मानवीय संवेदनाओं का विकास होना भी आधुनिक युग की एक अनिवार्य आवश्यकता है।", "సాంకేతిక విద్యతో పాటు మానవీయ संवेदनाల అభివృద్ధి కూడా ఆధునిక యుగంలో అనివార్యమైన అవసరం।"),
    ("पर्यावरण संरक्षण के प्रति जागरूकता फैलाना केवल सरकार का नहीं बल्कि प्रत्येक नागरिक का मौलिक कर्तव्य है।", "పర్యావరణ పరిరక్షణపై అవగాహన కల్పించడం కేవలం ప్రభుత్వానిదే కాదు, ప్రతి పౌరుడి ప్రాథమిక విధి।"),
    ("महिलाओं के आर्थिक सशक्तिकरण से न केवल परिवार की स्थिति सुधरती है बल्कि राष्ट्र का विकास भी तेज होता है।", "మహిళల ఆర్థిక సాధికారతతో కుటుంబ పరిస్థితి మెరుగుపడటమే కాకుండా దేశ అభివృద్ధి కూడా వేగవంతమవుతుంది।"),
    ("भ्रष्टाचार मुक्त समाज के निर्माण के लिए पारदर्शिता और जवाबदेही को हर स्तर पर लागू करना अनिवार्य है।", "అవినీతి రహిత సమాజం కోసం పారదర్శకత మరియు జవాబుదారీతనాన్ని ప్రతి స్థాయిలో అమలు చేయడం తప్పనిసరి।"),
    ("श्रम की गरिमा का सम्मान करना समाज के सर्वांगीण विकास के लिए एक महत्वपूर्ण सांस्कृतिक मूल्य है।", "శ్రమ గౌరవాన్ని గౌరవించడం సమాజ సర్వతోముఖాభివృద్ధికి ఒక ముఖ్యమైన సాంస్కృతిక విలువ।"),
    ("प्राकृतिक संसाधनों का विवेकपूर्ण उपयोग भविष्य की पीढ़ियों की आवश्यकताओं को सुरक्षित करने के लिए जरूरी है।", "భవిష్యత్ తరాల అవసరాలను సురక్షితం చేయడానికి సహజ వనరుల వివేకవంతమైన ఉపయోగం అవసరం।"),
    ("वैश्विक शांति की स्थापना के लिए विभिन्न राष्ट्रों के बीच सांस्कृतिक आदान-प्रदान और संवाद आवश्यक है।", "ప్రపంచ శాంతి స్థాపన కోసం వివిధ దేశాల మధ్య సాంస్కృతిక మార్పిడి మరియు సంభాషణ అవసరం।"),

    # --- VI. MEDICINE & BIOLOGY (51-70) ---
    ("प्रतिरक्षा प्रणाली शरीर को बाहरी बैक्टीरिया और वायरस के संक्रमण से बचाने वाली एक जटिल सुरक्षा व्यवस्था है।", "రోగనిరోధక వ్యవస్థ అనేది శరీరాన్ని బయటి బ్యాక్టీరియా మరియు వైరస్ల నుండి రక్షించే ఒక సంక్లిష్ట రక్షణ వ్యవస్థ।"),
    ("कैंसर अनुसंधान के क्षेत्र में इम्यूनोथेरेपी को एक क्रांतिकारी उपचार पद्धति के रूप में देखा जा रहा है।", "క్యాన్సర్ పరిశోధనా రంగంలో ఇమ్యునోథెరపీ ఒక విప్లవాత్మక చికిత్సా విధానంగా పరిగణించబడుతోంది।"),
    ("कोशिका विभाजन की अनियंत्रित प्रक्रिया ही ट्यूमर के विकास का मुख्य जैविक कारण मानी जाती है।", "కణ విభజన యొక్క అనియంత్రిత ప్రక్రియే ట్యూమర్ అభివృద్ధికి ప్రధాన జీవసంబంధ కారణం।"),
    ("हृदय रोगों के जोखिम को कम करने के लिए संतुलित आहार और नियमित व्यायाम का पालन करना अनिवार्य है।", "గుండె జబ్బుల ప్రమాదాన్ని తగ్గించడానికి సమతుల్య ఆహారం మరియు క్రమం తప్పకుండా వ్యాయామం చేయడం తప్పనిసరి।"),
    ("डीएनए अनुक्रमण तकनीक ने वंशानुगत बीमारियों की पहचान और उनके निवारण में अभूतपूर्व प्रगति की है।", "DNA సీక్వెన్సింగ్ సాంకేతికత వంశపారంపర్య వ్యాధుల గుర్తింపు మరియు నివారణలో అపూర్వమైన పురోగతిని సాధించింది।"),
    ("तंत्रिका तंत्र मस्तिष्क और शरीर के अन्य अंगों के बीच सूचनाओं के संचरण का समन्वय करता है।", "నరాల వ్యవస్థ మెదడు మరియు శరీరంలోని ఇతర భాగాల మధ్య సమాచార ప్రసారాన్ని సమన్వయం చేస్తుంది।"),
    ("एंटीबायोटिक दवाओं का अत्यधिक उपयोग भविष्य में दवाओं के प्रति प्रतिरोधक क्षमता विकसित होने का खतरा पैदा करता है।", "యాంటీబయాటిక్స్ అధికంగా వాడటం వల్ల భవిష్యత్తులో ఔషధ నిరోధకత పెరిగే ప్రమాదం ఉంది।"),
    ("रक्त शर्करा के स्तर को नियंत्रित करने के लिए अग्न्याशय द्वारा इंसुलिन हार्मोन का स्राव अत्यंत महत्वपूर्ण है।", "రక్తంలో చక్కెర స్థాయిలను నియంత్రించడానికి ప్యాంక్రియాస్ ఇన్సులిన్ హార్మోన్‌ను విడుదల చేయడం చాలా ముఖ్యం।"),
    ("विटामिन डी की कमी से हड्डियों की मजबूती प्रभावित होती है और ऑस्टियोपोरोसिस का खतरा बढ़ जाता है।", "విటమిన్ డి లోపం వల్ల ఎముకల బలం తగ్గుతుంది మరియు ఆస్టియోపోరోసిస్ ప్రమాదం పెరుగుతుంది।"),
    ("मानसिक स्वास्थ्य के प्रति सामाजिक दृष्टिकोण में बदलाव लाना चिकित्सा विज्ञान की एक बड़ी चुनौती है।", "మానసిక ఆరోగ్యం పట్ల సామాజిక దృక్పథంలో మార్పు తీసుకురావడం వైద్య విజ్ఞానానికి ఒక పెద్ద సవాలు।"),

    # --- VII. INFRASTRUCTURE & URBANIZATION (71-85) ---
    ("स्मार्ट सिटी मिशन का लक्ष्य शहरी क्षेत्रों में डिजिटल तकनीक के माध्यम से जीवन की गुणवत्ता में सुधार करना है।", "స్మార్ట్ సిటీ మిషన్ లక్ష్యం పట్టణ ప్రాంతాల్లో డిజిటల్ సాంకేతికత ద్వారా జీవన ప్రమాణాలను మెరుగుపరచడం।"),
    ("मेट्रो रेल परियोजनाओं ने बड़े शहरों में सार्वजनिक परिवहन की सुविधा और दक्षता को पूरी तरह से बदल दिया है।", "మెట్రో రైలు ప్రాజెక్టులు ప్రధాన నగరాల్లో ప్రజా రవాణా సౌకర్యాన్ని మరియు సామర్థ్యాన్ని పూర్తిగా మార్చేశాయి।"),
    ("सतत शहरीकरण के लिए अपशिष्ट प्रबंधन और जल पुनर्चक्रण प्रणालियों का विकास करना अनिवार्य हो गया है।", "సుస్థిర పట్టణీకరణ కోసం వ్యర్థాల నిర్వహణ మరియు నీటి పునర్వినియోగ వ్యవస్థల అభివృద్ధి అనివార్యమైంది।"),
    ("ग्रामीण बुनियादी ढांचे में सुधार से कृषि उत्पादों के विपणन और किसानों की आय में वृद्धि सुनिश्चित होती है।", "గ్రామీణ మౌలిక సదుపాయాల మెరుగుదల ద్వారా వ్యవసాయ ఉత్పత్తుల మార్కెటింగ్ మరియు రైతుల ఆదాయం పెరుగుతుంది।"),
    ("बंदरगाहों के आधुनिकीकरण से अंतरराष्ट्रीय व्यापार में लगने वाले समय और लागत को काफी कम किया जा सकता है।", "ఓడరేవుల ఆధునీకరణ ద్వారా అంతర్జాతీయ వాణిజ్యానికి పట్టే సమయాన్ని మరియు ఖర్చును గణనీయంగా తగ్గించవచ్చు।"),
    ("ऊर्जा सुरक्षा सुनिश्चित करने के लिए सौर और पवन ऊर्जा जैसे नवीकरणीय स्रोतों पर निर्भरता बढ़ानी होगी।", "ఇంధన భద్రతను నిర్ధారించడానికి సౌర మరియు పవన విద్యుత్ వంటి పునరుత్పాదక వనరులపై ఆధారపడటం పెంచాలి।"),
    ("पहाड़ी क्षेत्रों में सड़कों का निर्माण करते समय पारिस्थितिक संतुलन का ध्यान रखना एक बड़ी इंजीनियरिंग चुनौती है।", "కొండ ప్రాంతాలలో రోడ్ల నిర్మాణం చేపట్టేటప్పుడు పర్యావరణ సమతుల్యతను కాపాడటం ఒక పెద్ద ఇంజనీరింగ్ సవాలు।"),
    ("आवास योजनाओं का मुख्य उद्देश्य आर्थिक रूप से कमजोर वर्गों के लिए किफायती घर उपलब्ध कराना है।", "గృహనిర్మాణ పథకాల ప్రధాన ఉద్దేశ్యం ఆర్థికంగా వెనుకబడిన వర్గాలకు తక్కువ ధరకే ఇళ్లు అందించడం।"),
    ("डिजिटल कनेक्टिविटी ने दूरदराज के क्षेत्रों में शिक्षा और स्वास्थ्य सेवाओं की पहुँच को सुगम बना दिया है।", "డిజిటల్ కనెక్టివిటీ దూరప్రాంతాల్లో విద్య మరియు ఆరోగ్య సేవలు అందుబాటులోకి రావడాన్ని సులభతరం చేసింది।"),
    ("राष्ट्रीय राजमार्गों का नेटवर्क राज्यों के बीच वाणिज्यिक गतिविधियों और आर्थिक विकास की गति को बढ़ाता है।", "జాతీయ రహదారుల నెట్‌వర్క్ రాష్ట్రాల మధ్య వాణిజ్య కార్యకలాపాలను మరియు ఆర్థిక అభివృద్ధి వేగాన్ని పెంచుతుంది।"),

    # --- VIII. GENERAL COMPLEX TOPICS (86-100) ---
    ("वैश्वीकरण ने दुनिया को एक वैश्विक गांव में बदल दिया है जहाँ सूचनाओं का आदान-प्रदान तात्कालिक है।", "ప్రపంచీకరణ ప్రపంచాన్ని ఒక ప్రపంచ గ్రామంగా మార్చింది, ఇక్కడ సమాచార మార్పిడి తక్షణమే జరుగుతుంది।"),
    ("साहित्य समाज का दर्पण होता है जो प्रचलित सामाजिक मूल्यों और विसंगतियों को शब्दों में पिरोता है।", "సాహిత్యం సమాజానికి దర్పణం, ఇది ప్రబలమైన సామాజిక విలువలను మరియు వైరుధ్యాలను అక్షరబద్ధం చేస్తుంది।"),
    ("अनुशासन और निरंतरता ही वह कुंजी है जो किसी भी कठिन लक्ष्य को प्राप्त करने में सहायक होती है।", "క్రమశిక్షణ మరియు నిరంతర కృషి మాత్రమే ఏదైనా కష్టమైన లక్ష్యాన్ని సాధించడంలో సహాయపడతాయి।"),
    ("युवा शक्ति किसी भी राष्ट्र की सबसे बड़ी संपत्ति है जिसका सही दिशा में उपयोग करना आवश्यक है।", "యువశక్తి ఏ దేశానికైనా అతిపెద్ద ఆస్తి, దానిని సరైన దిశలో ఉపయోగించడం చాలా అవసరం।"),
    ("धैर्य और संयम विपत्ति के समय व्यक्ति के चरित्र की सबसे बड़ी परीक्षा लेते हैं।", "ధైర్యం మరియు సంయమనం కష్టకాలంలో వ్యక్తి యొక్క స్వభావాన్ని పరీక్షించే ప్రధాన అంశాలు।"),
    ("इतिहास हमें अतीत की गलतियों से सीखने और भविष्य को बेहतर बनाने की प्रेरणा देता है।", "చరిత్ర మనకు గత తప్పిదాల నుండి నేర్చుకోవడానికి మరియు భవిష్యత్తును మెరుగుపరుచుకోవడానికి స్ఫూర్తినిస్తుంది।"),
    ("पर्यटन क्षेत्र न केवल विदेशी मुद्रा अर्जित करता है बल्कि विभिन्न संस्कृतियों के बीच सेतु का कार्य करता है।", "పర్యాటక రంగం విదేశీ మారక ద్రవ్యాన్ని ఆర్జించడమే కాకుండా వివిధ సంస్కృతుల మధ్య వంతెనలా పనిచేస్తుంది।"),
    ("विज्ञान और अध्यात्म के बीच का संतुलन ही मनुष्य को पूर्णता की ओर ले जा सकता है।", "విజ్ఞానం మరియు ఆధ్యాత్మికత మధ్య సమతుల్యత మాత్రమే మనిషిని పరిపూర్ణత వైపు నడిపిస్తుంది।"),
    ("रचनात्मक लेखन के लिए कल्पनाशीलता और संवेदनशीलता का होना अनिवार्य गुण माना जाता है।", "సృజనాత్మక రచనకు ఊహాశక్తి మరియు సంవేదనాత్మకత ఉండటం అనివార్యమైన లక్షణాలు।"),
    ("खेलकूद न केवल शारीरिक स्वास्थ्य बल्कि टीम भावना और नेतृत्व क्षमता का भी विकास करते हैं।", "క్రీడలు కేవలం శారీరక ఆరోగ్యాన్నే కాకుండా జట్టు స్ఫూర్తిని మరియు నాయకత్వ పటిమను కూడా పెంచుతాయి।"),
    ("समय का प्रबंधन करने वाले व्यक्ति ही पेशेवर और व्यक्तिगत जीवन में सफलता प्राप्त करते हैं।", "సమయాన్ని సద్వినియోగం చేసుకునే వ్యక్తులు మాత్రమే వృత్తిపరమైన మరియు వ్యక్తిగత జీవితంలో విజయం సాధిస్తారు।"),
    ("सकारात्मक सोच मानसिक तनाव को कम करने और जीवन के प्रति उत्साह बढ़ाने में सहायक होती है।", "సానుకూల ఆలోచన మానసిక ఒత్తిడిని తగ్గించడానికి మరియు జీవితం పట్ల ఉత్సాహాన్ని పెంచడానికి సహాయపడుతుంది।"),
    ("आत्मविश्वास और कड़ी मेहनत का कोई विकल्प नहीं है जब बात सपनों को पूरा करने की हो।", "కలలను నెరవేర్చుకోవడంలో ఆత్మవిశ్వాసం మరియు కష్టపడటానికి మించిన ప్రత్యామ్నాయం లేదు।"),
    ("डिजिटल साक्षरता आज के युग में उतनी ही महत्वपूर्ण है जितनी कि पारंपरिक साक्षरता।", "సాంప్రదాయ అక్షరాస్యతతో పాటు డిజిటల్ అక్షరాస్యత కూడా నేటి యుగంలో అంతే ముఖ్యం।"),
    ("एक जागरूक नागरिक ही लोकतंत्र की जड़ों को मजबूत बनाने में महत्वपूर्ण भूमिका निभाता है।", "అవగాహన కలిగిన పౌరుడు మాత్రమే ప్రజాస్వామ్య మూలాలను బలోపేతం చేయడంలో కీలక పాత్ర పోషిస్తాడు।")
]

# --- EXECUTION ---
aligner = DeepAligner()
for i, (hi, te) in enumerate(COMPLEX_DATASET):
    links, score = aligner.align_complex(hi, te)
    print(f"\n--- SENTENCE {i+1} (Quality: {score:.4f}) ---")
    print(f"Hindi: {hi}\nTelugu: {te}\n")
    print(f"{'HINDI WORD':<20} | {'TELUGU WORD':<20} | {'CONFIDENCE'}")
    print("-" * 55)

    # Sort by sentence order (i) to see the flow
    for h_word, t_word, conf in sorted(links, key=lambda x: x[2], reverse=True):
        print(f"{h_word:<20} | {t_word:<20} | {conf:.5f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- SENTENCE 1 (Quality: 0.7758) ---
Hindi: भारतीय संविधान की प्रस्तावना में निहित धर्मनिरपेक्षता, समाजवाद और अखंडता के आदर्श राष्ट्र की एकता को सुदृढ़ करते हैं।
Telugu: భారత రాజ్యాంగ పీఠికలో పొందుపరచబడిన లౌకికవాదం, సామ్యవాదం మరియు అఖండత అనే ఆదర్శాలు దేశ ఐక్యతను బలోపేతం చేస్తాయి।

HINDI WORD           | TELUGU WORD          | CONFIDENCE
-------------------------------------------------------
भारतीय               | భారత                 | 1.00000
संविधान              | రాజ్యాంగ             | 1.00000
की                   | పీఠికలో              | 1.00000
निहित                | పొందుపరచబడిన         | 1.00000
समाजवाद              | సామ్యవాదం            | 1.00000
और                   | మరియు                | 1.00000
आदर्श                | ఆదర్శాలు             | 1.00000
राष्ट्र              | దేశ                  | 1.00000
एकता                 | ఐక్యతను              | 1.00000
सुदृढ़               | బలోపేతం              | 1.00000
करते                 | చేస్తాయి             | 1.00000
अखंडता    

In [ ]:
import torch
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from torch.nn.functional import softmax

class DeepAligner:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = SentenceTransformer('sentence-transformers/LaBSE').to(self.device)
        self.tokenizer = self.model.tokenizer

    def _get_vectors(self, tokens):
        inputs = self.tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.model[0].auto_model(**inputs, output_hidden_states=True)
            # Using Layer 8 for deep semantic features
            embeddings = outputs.hidden_states[8][0]

        word_ids = inputs.word_ids(batch_index=0)
        return torch.stack([embeddings[[j for j, w_id in enumerate(word_ids) if w_id == i]].mean(dim=0)
                            for i in range(len(tokens))])

    def align_complex(self, hi_text, te_text):
        hi_toks = re.sub(r'[।\?\!\,\(\)"]', '', hi_text).strip().split()
        te_toks = re.sub(r'[।\?\!\,\(\)"]', '', te_text).strip().split()

        hi_vecs = self._get_vectors(hi_toks)
        te_vecs = self._get_vectors(te_toks)

        # Compute Double Softmax with High Temperature for Precision
        sim_mat = torch.matmul(hi_vecs, te_vecs.T)
        mutual_conf = (softmax(sim_mat * 12.0, dim=1) * softmax(sim_mat * 12.0, dim=0)).cpu().numpy()

        links = []
        for i in range(len(hi_toks)):
            for j in range(len(te_toks)):
                if mutual_conf[i, j] > 0.01: # Lower threshold to see ALL connections
                    links.append((hi_toks[i], te_toks[j], mutual_conf[i, j]))

        avg_conf = np.mean([l[2] for l in links]) if links else 0
        coverage = len(links) / max(len(hi_toks), len(te_toks))
        return links, (avg_conf + coverage) / 2

# --- LONG & COMPLEX DATASET SAMPLES (Expand to 100) ---
# 100 COMPLEX HINDI-TELUGU SENTENCE PAIRS
# Format: (Hindi_Sentence, Telugu_Sentence)

COMPLEX_DATASET = [
    # --- I. LEGAL & GOVERNANCE ---
    ("भारतीय संविधान की प्रस्तावना में निहित धर्मनिरपेक्षता, समाजवाद और अखंडता के आदर्श राष्ट्र की एकता को सुदृढ़ करते हैं।", "భారత రాజ్యాంగ పీఠికలో పొందుపరచబడిన లౌకికవాదం, సామ్యవాదం మరియు అఖండత అనే ఆదర్శాలు దేశ ఐక్యతను బలోపేతం చేస్తాయి।"),
    ("उच्चतम न्यायालय ने निर्णय दिया कि निजता का अधिकार संविधान के अनुच्छेद 21 के तहत जीवन और व्यक्तिगत स्वतंत्रता का अभिन्न अंग है।", "గోప్యత హక్కు రాజ్యాంగంలోని 21వ అధికరణం ప్రకారం జీవించే మరియు వ్యక్తిగత స్వేచ్ఛలో అంతర్భాగమని సుప్రీంకోర్టు తీర్పునిచ్చింది।"),
    ("द्विसदनीय विधायिका की संरचना यह सुनिश्चित करती है कि केंद्र और राज्यों के बीच शक्तियों का वितरण संतुलन में बना रहे।", "ద్విసభా శాసనసభ నిర్మాణం కేంద్రం మరియు రాష్ట్రాల మధ్య అధికారాల పంపిణీ సమతుల్యంగా ఉండేలా చూస్తుంది।"),
    ("चुनाव आयोग एक स्वायत्त संवैधानिक प्राधिकरण है जो भारत में संघ और राज्य चुनाव प्रक्रियाओं के प्रशासन के लिए उत्तरदायी है।", "ఎన్నికల సంఘం అనేది భారతదేశంలో కేంద్ర మరియు రాష్ట్ర ఎన్నికల ప్రక్రియల నిర్వహణకు బాధ్యత వహించే ఒక స్వయంప్రతిపత్తి కలిగిన రాజ్యాంగ సంస్థ।"),
    ("लोकपाल और लोकायुक्त अधिनियम का उद्देश्य सार्वजनिक पदाधिकारियों के विरुद्ध भ्रष्टाचार के आरोपों की निष्पक्ष जांच करना है।", "లోక్‌పాల్ మరియు లోకాయుక్త చట్టం యొక్క ఉద్దేశ్యం ప్రభుత్వ అధికారులపై వచ్చే అవినీతి ఆరోపణలపై నిష్పక్షపాతంగా విచారణ జరపడం।"),
    ("मौलिक कर्तव्यों का पालन करना प्रत्येक नागरिक का नैतिक उत्तरदायित्व है ताकि समाज में सद्भाव और अनुशासन बना रहे।", "సమాజంలో సామరస్యం మరియు క్రమశిక్షణను కాపాడటానికి ప్రతి పౌరుడు ప్రాథమిక విధులను పాటించడం నైతిక బాధ్యత।"),
    ("संसद के दोनों सदनों द्वारा पारित विधेयक को राष्ट्रपति की सहमति प्राप्त होने के बाद ही आधिकारिक कानून माना जाता है।", "పార్లమెంటు ఉభయ సభలు ఆమోదించిన బిల్లుకు రాష్ట్రపతి అంగీకారం లభించిన తర్వాతే అది అధికారిక చట్టంగా మారుతుంది।"),
    ("प्रशासनिक विकेंद्रीकरण का मुख्य उद्देश्य स्थानीय स्वशासन संस्थानों को सशक्त बनाना और जमीनी स्तर पर विकास सुनिश्चित करना है।", "పరిపాలనా వికేంద్రీకరణ యొక్క ప్రధాన ఉద్దేశ్యం స్థానిక స్వపరిపాలన సంస్థలను బలోపేతం చేయడం మరియు క్షేత్రస్థాయిలో అభివృద్ధిని నిర్ధారించడం।"),
    ("न्यायिक सक्रियता के माध्यम से अदालतें अक्सर कार्यपालिका को उनके संवैधानिक कर्तव्यों के प्रति सचेत करने का कार्य करती हैं।", "న్యాయవ్యవస్థ క్రియాశీలత ద్వారా కోర్టులు తరచుగా కార్యనిర్వాహక వ్యవస్థను వారి రాజ్యాంగ విధులను గుర్తు చేస్తూ ఉంటాయి।"),
    ("अन्तरराष्ट्रीय न्यायालय का मुख्य कार्य राज्यों के बीच कानूनी विवादों को सुलझाना और कानूनी प्रश्नों पर सलाह देना है।", "అంతర్జాతీయ న్యాయస్థానం యొక్క ప్రధాన విధి దేశాల మధ్య చట్టపరమైన వివాదాలను పరిష్కరించడం మరియు చట్టపరమైన ప్రశ్నలపై సలహాలు ఇవ్వడం।"),

    # --- II. SPACE & PHYSICS ---
    ("भारतीय अंतरिक्ष अनुसंधान संगठन ने मंगलयान मिशन के सफल प्रक्षेपण के साथ अंतरग्रहीय अन्वेषण में नया अध्याय लिखा।", "భారత అంతరిక్ష పరిశోధనా సంస్థ మంగళయాన్ మిషన్ విజయవంత ప్రయోగంతో అంతర గ్రహ పరిశోధనలో కొత్త అధ్యాయాన్ని లిఖించింది।"),
    ("क्वांटम भौतिकी के सिद्धांत बताते हैं कि सूक्ष्म स्तर पर कणों का व्यवहार शास्त्रीय भौतिकी के नियमों से भिन्न होता है।", "క్వాంటం భౌతికశాస్త్ర సూత్రాలు సూక్ష్మ స్థాయిలో కణాల ప్రవర్తన శాస్త్రీయ భౌతికశాస్త్ర నియమాల కంటే భిన్నంగా ఉంటుందని చెబుతాయి।"),
    ("पारिस्थितिक तंत्र में जैव विविधता की हानि खाद्य श्रृंखला के स्थायित्व को गंभीर रूप से असंतुलित कर सकती है।", "పర్యావరణ వ్యవస్థలో జీవవైవిధ్యం తగ్గడం వల్ల ఆహార గొలుసు యొక్క స్థిరత్వం తీవ్రంగా దెబ్బతింటుంది।"),
    ("वैश्विक तापन के कारण ग्लेशियरों का पिघलना समुद्र के स्तर में वृद्धि और तटीय क्षेत्रों के जलमग्न होने का कारण है।", "భూతాపం వల్ల హిమానీనదాలు కరగడం సముద్ర మట్టాలు పెరగడానికి మరియు తీర ప్రాంతాలు మునిగిపోవడానికి కారణం।"),
    ("प्रकाश की दोहरी प्रकृति यह सिद्ध करती है कि वह तरंग और कण दोनों के रूप में व्यवहार कर सकता है।", "కాంతి యొక్క ద్వంద్వ స్వభావం అది తరంగం మరియు కణం రెండింటి రూపంలో ప్రవర్తించగలదని నిరూపిస్తుంది।"),
    ("ब्लैक होल अंतरिक्ष में वह स्थान है जहाँ गुरुत्वाकर्षण इतना प्रबल होता है कि प्रकाश भी वहां से नहीं निकल सकता।", "బ్లాక్ హోల్ అనేది అంతరిక్షంలో గురుత్వాకర్షణ శక్తి ఎంత బలంగా ఉంటుందంటే అక్కడ నుండి కాంతి కూడా తప్పించుకోలేదు।"),
    ("सौरमंडल के ग्रहों की गति को समझने के लिए केप्लर के नियमों का अध्ययन करना खगोल विज्ञान में अनिवार्य है।", "సౌర వ్యవస్థలోని గ్రహాల గమనాన్ని అర్థం చేసుకోవడానికి కెప్లర్ నియమాలను అధ్యయనం చేయడం ఖగోళ శాస్త్రంలో తప్పనిసరి।"),
    ("परमाणु रिएक्टरों में नियंत्रित नाभिकीय विखंडन की प्रक्रिया के माध्यम से भारी मात्रा में ऊर्जा उत्पन्न की जाती है।", "అణు రియాక్టర్లలో నియంత్రిత కేంద్రక విచ్ఛిత్తి ప్రక్రియ ద్వారా భారీ మొత్తంలో ఇంధనం ఉత్పత్తి చేయబడుతుంది।"),
    ("ओजोन परत वायुमंडल के ऊपरी भाग में स्थित है जो हानिकारक पराबैंगनी विकिरण को पृथ्वी तक पहुँचने से रोकती है।", "ఓజోన్ పొర వాతావరణం పైభాగంలో ఉండి హానికరమైన అతినీలలోహిత వికిరణం భూమికి చేరకుండా అడ్డుకుంటుంది।"),
    ("आइंस्टीन का सापेक्षता का सिद्धांत स्थान और समय के बीच के संबंध को वैज्ञानिक रूप से पुनर्परिभाषित करता है।", "ఐన్‌స్టీన్ సాపేక్షత సిద్ధాంతం స్థలం మరియు కాలం మధ్య సంబంధాన్ని శాస్త్రీయంగా పునర్నిర్వచించింది।"),

    # --- III. TECHNOLOGY & AI ---
    ("कृत्रिम बुद्धिमत्ता के एल्गोरिदम डेटा के विशाल भंडार का विश्लेषण करके भविष्य के रुझानों की सटीक भविष्यवाणी करते हैं।", "కృత్రిమ మేధస్సు అల్గారిథమ్స్ భారీ డేటాను విశ్లేషించి భవిష్యత్తు ధోరణులను ఖచ్చితంగా అంచనా వేస్తాయి।"),
    ("ब्लॉकचेन तकनीक विकेंद्रीकृत बहीखाता प्रणाली के माध्यम से वित्तीय लेनदेन की सुरक्षा और पारदर्शिता सुनिश्चित करती है।", "బ్లాక్‌చైన్ సాంకేతికత వికేంద్రీకృత లెడ్జర్ వ్యవస్థ ద్వారా ఆర్థిక లావాదేవీల భద్రత మరియు పారదర్శకతను నిర్ధారిస్తుంది।"),
    ("वस्तु एवं सेवा कर (GST) के कार्यान्वयन ने भारत में अप्रत्यक्ष कर ढांचे को एकीकृत करके साझा बाजार बनाया है।", "వస్తు సేవల పన్ను (GST) అమలు భారతదేశంలో పరోక్ష పన్ను నిర్మాణాన్ని ఏకీకృతం చేయడం ద్వారా ఉమ్మడి మార్కెట్‌ను సృష్టించింది।"),
    ("सूक्ष्म, लघु और मध्यम उद्यम क्षेत्र भारतीय अर्थव्यवस्था की रीढ़ है जो ग्रामीण रोजगार सृजन में योगदान देता है।", "సూక్ష్మ, చిన్న మరియు మధ్య తరహా పరిశ్రమల రంగం భారత ఆర్థిక వ్యవస్థకు వెన్నెముక, ఇది గ్రామీణ ఉపాధి కల్పనలో తోడ్పడుతుంది।"),
    ("साइबर सुरक्षा की चुनौतियाँ निरंतर बढ़ती जा रही हैं क्योंकि डिजिटल बुनियादी ढाँचा अधिक जटिल होता जा रहा है।", "డిజిటల్ మౌలిక సదుపాయాలు మరింత సంక్లిష్టంగా మారుతున్న కొద్దీ సైబర్ భద్రతా సవాళ్లు నిరంతరం పెరుగుతున్నాయి।"),
    ("मशीन लर्निंग के मॉडल जितना अधिक डेटा प्राप्त करते हैं, उनकी सटीकता और प्रदर्शन में उतना ही सुधार होता है।", "మెషిన్ లెర్నింగ్ మోడల్స్ ఎంత ఎక్కువ డేటాను పొందితే, వాటి ఖచ్చితత్వం మరియు పనితీరు అంత మెరుగుపడుతుంది।"),
    ("क्लाउड कंप्यूटिंग व्यवसायों को भौतिक बुनियादी ढांचे में निवेश किए बिना बड़े पैमाने पर डेटा स्टोर करने की अनुमति देती है।", "క్లౌడ్ కంప్యూటింగ్ భౌతిక మౌలిక సదుపాయాలపై పెట్టుబడి పెట్టకుండానే భారీ డేటాను భద్రపరిచేందుకు వ్యాపారాలకు అనుమతిస్తుంది।"),
    ("इंटरनेट ऑफ थिंग्स (IoT) विभिन्न उपकरणों को एक नेटवर्क से जोड़कर स्मार्ट स्वचालन की सुविधा प्रदान करता है।", "ఇంటర్నెట్ ఆఫ్ థింగ్స్ (IoT) వివిధ పరికరాలను నెట్‌వర్క్‌తో అనుసంధానించడం ద్వారా స్మార్ట్ ఆటోమేషన్‌కు వీలు కల్పిస్తుంది।"),
    ("स्वदेशी विनिर्माण को बढ़ावा देने के लिए 'मेक इन इंडिया' पहल ने रक्षा उत्पादन क्षेत्र में निवेश को आकर्षित किया है।", "స్వదేశీ తయారీని ప్రోత్సహించడానికి 'మేక్ ఇన్ ఇండియా' చొరవ రక్షణ ఉత్పత్తి రంగంలో పెట్టుబడులను ఆకర్షించింది।"),
    ("आर्थिक मंदी के दौरान सरकारें अक्सर तरलता बढ़ाने के लिए ब्याज दरों को कम करने जैसे राजकोषीय उपाय करती हैं।", "ఆర్థిక మాంద్యం సమయంలో ప్రభుత్వం లిక్విడిటీ పెంచడానికి వడ్డీ రేట్లను తగ్గించడం వంటి చర్యలు తీసుకుంటుంది।"),

    # --- IV. PHILOSOPHY & HISTORY ---
    ("अद्वैत वेदांत दर्शन के अनुसार आत्मा और परमात्मा के बीच का भेद केवल अविद्या या अज्ञान के कारण प्रतीत होता है।", "అద్వైత వేదాంత దర్శనం ప్రకారం ఆత్మ మరియు పరమాత్మ మధ్య భేదం కేవలం అవిద్య లేదా అజ్ఞానం వల్లనే కనిపిస్తుంది।"),
    ("मौर्य साम्राज्य के शिलालेख राजा अशोक की धम्म नीति और लोक कल्याणकारी प्रशासन के जीवंत प्रमाण प्रस्तुत करते हैं।", "మౌర్య సామ్రాజ్య శిలాశాసనాలు అశోక చక్రవర్తి ధమ్మ నీతి మరియు ప్రజా సంక్షేమ పాలనకు సజీవ సాక్ష్యాలుగా నిలుస్తాయి।"),
    ("सांस्कृतिक बहुलवाद भारतीय समाज की वह विशेषता है जो विभिन्न भाषाओं और परंपराओं को एक सूत्र में पिरोती है।", "సాంస్కృతిక బహుళత్వం అనేది భారతీయ సమాజం యొక్క ప్రత్యేకత, ఇది వివిధ భాషలను మరియు సంప్రదాయాలను ఒకే త్రాడుపైకి తెస్తుంది।"),
    ("पुनर्जागरण काल ने कला और विज्ञान के क्षेत्र में तर्क और मानवतावाद को धार्मिक कट्टरता पर वरीयता दी।", "పునర్జీవన కాలం కళ మరియు విజ్ఞాన రంగాలలో మతపరమైన ఛాందసవాదం కంటే తర్కానికి మరియు మానవతావాదానికి ప్రాధాన్యతనిచ్చింది।"),
    ("प्राचीन भारतीय चिकित्सा पद्धति आयुर्वेद रोगों के उपचार के बजाय जीवनशैली के संतुलन पर अधिक बल देती है।", "ప్రాచీన భారతీయ వైద్య విధానం ఆయుర్వేదం వ్యాధుల చికిత్స కంటే జీవనశైలి సమతుల్యతపై ఎక్కువ మొగ్గు చూపుతుంది।"),
    ("बुद्ध के अष्टांगिक मार्ग का उद्देश्य मानवीय दुखों का अंत करना और निर्वाण की प्राप्ति के लिए मार्गदर्शन करना है।", "బుద్ధుని అష్టాంగ మార్గం మానవ దుఃఖాలను అంతం చేయడానికి మరియు నిర్వాణం సాధించడానికి మార్గదర్శకత్వం చేస్తుంది।"),
    ("सिंधु घाटी सभ्यता की शहरी योजना और जल निकासी प्रणाली उस समय की उन्नत तकनीकी समझ को दर्शाती है।", "సింధు లోయ నాగరికత యొక్క పట్టణ ప్రణాళిక మరియు డ్రైనేజీ వ్యవస్థ అప్పటి అధునాతన సాంకేతిక అవగాహనను ప్రతిబింబిస్తాయి।"),
    ("भक्ति आंदोलन ने जातिगत बंधनों को तोड़कर ईश्वर के प्रति व्यक्तिगत समर्पण और प्रेम के मार्ग को लोकप्रिय बनाया।", "భక్తి ఉద్యమం కుల బంధాలను తెంచి భగవంతుని పట్ల వ్యక్తిగత అంకితభావం మరియు ప్రేమ మార్గాన్ని ప్రాచుర్యంలోకి తెచ్చింది।"),
    ("गांधीवादी दर्शन में सत्य और अहिंसा को केवल आदर्श नहीं बल्कि सामाजिक परिवर्तन के अस्त्र के रूप में देखा गया है।", "గాంధేయ దర్శనంలో సత్యం మరియు అహింస కేవలం ఆదర్శాలు మాత్రమే కాదు, సామాజిక మార్పుకు అస్త్రాలుగా చూడబడ్డాయి।"),
    ("मुगल वास्तुकला में भारतीय और फारसी शैलियों का अद्भुत समन्वय ताजमहल जैसी भव्य इमारतों में परिलक्षित होता है।", "మొఘల్ వాస్తుశిల్పంలో భారతీయ మరియు పర్షియన్ శైలుల అద్భుత సమన్వయం తాజ్ మహల్ వంటి భవనాలలో కనిపిస్తుంది।"),

    # --- V. EDUCATION & ETHICS (41-60) ---
    ("नई शिक्षा नीति का उद्देश्य छात्रों में रटने की प्रवृत्ति के बजाय आलोचनात्मक सोच और रचनात्मकता विकसित करना है।", "కొత్త విద్యా విధానం విద్యార్థులలో బట్టీ పట్టే ధోరణి కంటే విమర్శనాత్మక ఆలోచన మరియు సృజనాత్మకతను పెంపొందించడం లక్ష్యంగా పెట్టుకుంది।"),
    ("समावेशी विकास का अर्थ है कि आर्थिक प्रगति का लाभ समाज के सबसे अंतिम पायदान पर खड़े व्यक्ति तक पहुँचे।", "సమ్మిళిత వృద్ధి అంటే ఆర్థిక పురోగతి యొక్క ప్రయోజనాలు సమాజంలోని చివరి వ్యక్తికి కూడా అందడం।"),
    ("नैतिक मूल्य ही वह आधार हैं जिस पर एक स्वस्थ और न्यायपूर्ण समाज की नींव रखी जा सकती है।", "నైతిక విలువల ప్రాతిపదికన మాత్రమే ఆరోగ్యకరమైన మరియు న్యాయమైన సమాజ పునాది నిర్మించబడుతుంది।"),
    ("तकनीकी शिक्षा के साथ-साथ मानवीय संवेदनाओं का विकास होना भी आधुनिक युग की एक अनिवार्य आवश्यकता है।", "సాంకేతిక విద్యతో పాటు మానవీయ संवेदनाల అభివృద్ధి కూడా ఆధునిక యుగంలో అనివార్యమైన అవసరం।"),
    ("पर्यावरण संरक्षण के प्रति जागरूकता फैलाना केवल सरकार का नहीं बल्कि प्रत्येक नागरिक का मौलिक कर्तव्य है।", "పర్యావరణ పరిరక్షణపై అవగాహన కల్పించడం కేవలం ప్రభుత్వానిదే కాదు, ప్రతి పౌరుడి ప్రాథమిక విధి।"),
    ("महिलाओं के आर्थिक सशक्तिकरण से न केवल परिवार की स्थिति सुधरती है बल्कि राष्ट्र का विकास भी तेज होता है।", "మహిళల ఆర్థిక సాధికారతతో కుటుంబ పరిస్థితి మెరుగుపడటమే కాకుండా దేశ అభివృద్ధి కూడా వేగవంతమవుతుంది।"),
    ("भ्रष्टाचार मुक्त समाज के निर्माण के लिए पारदर्शिता और जवाबदेही को हर स्तर पर लागू करना अनिवार्य है।", "అవినీతి రహిత సమాజం కోసం పారదర్శకత మరియు జవాబుదారీతనాన్ని ప్రతి స్థాయిలో అమలు చేయడం తప్పనిసరి।"),
    ("श्रम की गरिमा का सम्मान करना समाज के सर्वांगीण विकास के लिए एक महत्वपूर्ण सांस्कृतिक मूल्य है।", "శ్రమ గౌరవాన్ని గౌరవించడం సమాజ సర్వతోముఖాభివృద్ధికి ఒక ముఖ్యమైన సాంస్కృతిక విలువ।"),
    ("प्राकृतिक संसाधनों का विवेकपूर्ण उपयोग भविष्य की पीढ़ियों की आवश्यकताओं को सुरक्षित करने के लिए जरूरी है।", "భవిష్యత్ తరాల అవసరాలను సురక్షితం చేయడానికి సహజ వనరుల వివేకవంతమైన ఉపయోగం అవసరం।"),
    ("वैश्विक शांति की स्थापना के लिए विभिन्न राष्ट्रों के बीच सांस्कृतिक आदान-प्रदान और संवाद आवश्यक है।", "ప్రపంచ శాంతి స్థాపన కోసం వివిధ దేశాల మధ్య సాంస్కృతిక మార్పిడి మరియు సంభాషణ అవసరం।"),

    # --- VI. MEDICINE & BIOLOGY (51-70) ---
    ("प्रतिरक्षा प्रणाली शरीर को बाहरी बैक्टीरिया और वायरस के संक्रमण से बचाने वाली एक जटिल सुरक्षा व्यवस्था है।", "రోగనిరోధక వ్యవస్థ అనేది శరీరాన్ని బయటి బ్యాక్టీరియా మరియు వైరస్ల నుండి రక్షించే ఒక సంక్లిష్ట రక్షణ వ్యవస్థ।"),
    ("कैंसर अनुसंधान के क्षेत्र में इम्यूनोथेरेपी को एक क्रांतिकारी उपचार पद्धति के रूप में देखा जा रहा है।", "క్యాన్సర్ పరిశోధనా రంగంలో ఇమ్యునోథెరపీ ఒక విప్లవాత్మక చికిత్సా విధానంగా పరిగణించబడుతోంది।"),
    ("कोशिका विभाजन की अनियंत्रित प्रक्रिया ही ट्यूमर के विकास का मुख्य जैविक कारण मानी जाती है।", "కణ విభజన యొక్క అనియంత్రిత ప్రక్రియే ట్యూమర్ అభివృద్ధికి ప్రధాన జీవసంబంధ కారణం।"),
    ("हृदय रोगों के जोखिम को कम करने के लिए संतुलित आहार और नियमित व्यायाम का पालन करना अनिवार्य है।", "గుండె జబ్బుల ప్రమాదాన్ని తగ్గించడానికి సమతుల్య ఆహారం మరియు క్రమం తప్పకుండా వ్యాయామం చేయడం తప్పనిసరి।"),
    ("डीएनए अनुक्रमण तकनीक ने वंशानुगत बीमारियों की पहचान और उनके निवारण में अभूतपूर्व प्रगति की है।", "DNA సీక్వెన్సింగ్ సాంకేతికత వంశపారంపర్య వ్యాధుల గుర్తింపు మరియు నివారణలో అపూర్వమైన పురోగతిని సాధించింది।"),
    ("तंत्रिका तंत्र मस्तिष्क और शरीर के अन्य अंगों के बीच सूचनाओं के संचरण का समन्वय करता है।", "నరాల వ్యవస్థ మెదడు మరియు శరీరంలోని ఇతర భాగాల మధ్య సమాచార ప్రసారాన్ని సమన్వయం చేస్తుంది।"),
    ("एंटीबायोटिक दवाओं का अत्यधिक उपयोग भविष्य में दवाओं के प्रति प्रतिरोधक क्षमता विकसित होने का खतरा पैदा करता है।", "యాంటీబయాటిక్స్ అధికంగా వాడటం వల్ల భవిష్యత్తులో ఔషధ నిరోధకత పెరిగే ప్రమాదం ఉంది।"),
    ("रक्त शर्करा के स्तर को नियंत्रित करने के लिए अग्न्याशय द्वारा इंसुलिन हार्मोन का स्राव अत्यंत महत्वपूर्ण है।", "రక్తంలో చక్కెర స్థాయిలను నియంత్రించడానికి ప్యాంక్రియాస్ ఇన్సులిన్ హార్మోన్‌ను విడుదల చేయడం చాలా ముఖ్యం।"),
    ("विटामिन डी की कमी से हड्डियों की मजबूती प्रभावित होती है और ऑस्टियोपोरोसिस का खतरा बढ़ जाता है।", "విటమిన్ డి లోపం వల్ల ఎముకల బలం తగ్గుతుంది మరియు ఆస్టియోపోరోసిస్ ప్రమాదం పెరుగుతుంది।"),
    ("मानसिक स्वास्थ्य के प्रति सामाजिक दृष्टिकोण में बदलाव लाना चिकित्सा विज्ञान की एक बड़ी चुनौती है।", "మానసిక ఆరోగ్యం పట్ల సామాజిక దృక్పథంలో మార్పు తీసుకురావడం వైద్య విజ్ఞానానికి ఒక పెద్ద సవాలు।"),

    # --- VII. INFRASTRUCTURE & URBANIZATION (71-85) ---
    ("स्मार्ट सिटी मिशन का लक्ष्य शहरी क्षेत्रों में डिजिटल तकनीक के माध्यम से जीवन की गुणवत्ता में सुधार करना है।", "స్మార్ట్ సిటీ మిషన్ లక్ష్యం పట్టణ ప్రాంతాల్లో డిజిటల్ సాంకేతికత ద్వారా జీవన ప్రమాణాలను మెరుగుపరచడం।"),
    ("मेट्रो रेल परियोजनाओं ने बड़े शहरों में सार्वजनिक परिवहन की सुविधा और दक्षता को पूरी तरह से बदल दिया है।", "మెట్రో రైలు ప్రాజెక్టులు ప్రధాన నగరాల్లో ప్రజా రవాణా సౌకర్యాన్ని మరియు సామర్థ్యాన్ని పూర్తిగా మార్చేశాయి।"),
    ("सतत शहरीकरण के लिए अपशिष्ट प्रबंधन और जल पुनर्चक्रण प्रणालियों का विकास करना अनिवार्य हो गया है।", "సుస్థిర పట్టణీకరణ కోసం వ్యర్థాల నిర్వహణ మరియు నీటి పునర్వినియోగ వ్యవస్థల అభివృద్ధి అనివార్యమైంది।"),
    ("ग्रामीण बुनियादी ढांचे में सुधार से कृषि उत्पादों के विपणन और किसानों की आय में वृद्धि सुनिश्चित होती है।", "గ్రామీణ మౌలిక సదుపాయాల మెరుగుదల ద్వారా వ్యవసాయ ఉత్పత్తుల మార్కెటింగ్ మరియు రైతుల ఆదాయం పెరుగుతుంది।"),
    ("बंदरगाहों के आधुनिकीकरण से अंतरराष्ट्रीय व्यापार में लगने वाले समय और लागत को काफी कम किया जा सकता है।", "ఓడరేవుల ఆధునీకరణ ద్వారా అంతర్జాతీయ వాణిజ్యానికి పట్టే సమయాన్ని మరియు ఖర్చును గణనీయంగా తగ్గించవచ్చు।"),
    ("ऊर्जा सुरक्षा सुनिश्चित करने के लिए सौर और पवन ऊर्जा जैसे नवीकरणीय स्रोतों पर निर्भरता बढ़ानी होगी।", "ఇంధన భద్రతను నిర్ధారించడానికి సౌర మరియు పవన విద్యుత్ వంటి పునరుత్పాదక వనరులపై ఆధారపడటం పెంచాలి।"),
    ("पहाड़ी क्षेत्रों में सड़कों का निर्माण करते समय पारिस्थितिक संतुलन का ध्यान रखना एक बड़ी इंजीनियरिंग चुनौती है।", "కొండ ప్రాంతాలలో రోడ్ల నిర్మాణం చేపట్టేటప్పుడు పర్యావరణ సమతుల్యతను కాపాడటం ఒక పెద్ద ఇంజనీరింగ్ సవాలు।"),
    ("आवास योजनाओं का मुख्य उद्देश्य आर्थिक रूप से कमजोर वर्गों के लिए किफायती घर उपलब्ध कराना है।", "గృహనిర్మాణ పథకాల ప్రధాన ఉద్దేశ్యం ఆర్థికంగా వెనుకబడిన వర్గాలకు తక్కువ ధరకే ఇళ్లు అందించడం।"),
    ("डिजिटल कनेक्टिविटी ने दूरदराज के क्षेत्रों में शिक्षा और स्वास्थ्य सेवाओं की पहुँच को सुगम बना दिया है।", "డిజిటల్ కనెక్టివిటీ దూరప్రాంతాల్లో విద్య మరియు ఆరోగ్య సేవలు అందుబాటులోకి రావడాన్ని సులభతరం చేసింది।"),
    ("राष्ट्रीय राजमार्गों का नेटवर्क राज्यों के बीच वाणिज्यिक गतिविधियों और आर्थिक विकास की गति को बढ़ाता है।", "జాతీయ రహదారుల నెట్‌వర్క్ రాష్ట్రాల మధ్య వాణిజ్య కార్యకలాపాలను మరియు ఆర్థిక అభివృద్ధి వేగాన్ని పెంచుతుంది।"),

    # --- VIII. GENERAL COMPLEX TOPICS (86-100) ---
    ("वैश्वीकरण ने दुनिया को एक वैश्विक गांव में बदल दिया है जहाँ सूचनाओं का आदान-प्रदान तात्कालिक है।", "ప్రపంచీకరణ ప్రపంచాన్ని ఒక ప్రపంచ గ్రామంగా మార్చింది, ఇక్కడ సమాచార మార్పిడి తక్షణమే జరుగుతుంది।"),
    ("साहित्य समाज का दर्पण होता है जो प्रचलित सामाजिक मूल्यों और विसंगतियों को शब्दों में पिरोता है।", "సాహిత్యం సమాజానికి దర్పణం, ఇది ప్రబలమైన సామాజిక విలువలను మరియు వైరుధ్యాలను అక్షరబద్ధం చేస్తుంది।"),
    ("अनुशासन और निरंतरता ही वह कुंजी है जो किसी भी कठिन लक्ष्य को प्राप्त करने में सहायक होती है।", "క్రమశిక్షణ మరియు నిరంతర కృషి మాత్రమే ఏదైనా కష్టమైన లక్ష్యాన్ని సాధించడంలో సహాయపడతాయి।"),
    ("युवा शक्ति किसी भी राष्ट्र की सबसे बड़ी संपत्ति है जिसका सही दिशा में उपयोग करना आवश्यक है।", "యువశక్తి ఏ దేశానికైనా అతిపెద్ద ఆస్తి, దానిని సరైన దిశలో ఉపయోగించడం చాలా అవసరం।"),
    ("धैर्य और संयम विपत्ति के समय व्यक्ति के चरित्र की सबसे बड़ी परीक्षा लेते हैं।", "ధైర్యం మరియు సంయమనం కష్టకాలంలో వ్యక్తి యొక్క స్వభావాన్ని పరీక్షించే ప్రధాన అంశాలు।"),
    ("इतिहास हमें अतीत की गलतियों से सीखने और भविष्य को बेहतर बनाने की प्रेरणा देता है।", "చరిత్ర మనకు గత తప్పిదాల నుండి నేర్చుకోవడానికి మరియు భవిష్యత్తును మెరుగుపరుచుకోవడానికి స్ఫూర్తినిస్తుంది।"),
    ("पर्यटन क्षेत्र न केवल विदेशी मुद्रा अर्जित करता है बल्कि विभिन्न संस्कृतियों के बीच सेतु का कार्य करता है।", "పర్యాటక రంగం విదేశీ మారక ద్రవ్యాన్ని ఆర్జించడమే కాకుండా వివిధ సంస్కృతుల మధ్య వంతెనలా పనిచేస్తుంది।"),
    ("विज्ञान और अध्यात्म के बीच का संतुलन ही मनुष्य को पूर्णता की ओर ले जा सकता है।", "విజ్ఞానం మరియు ఆధ్యాత్మికత మధ్య సమతుల్యత మాత్రమే మనిషిని పరిపూర్ణత వైపు నడిపిస్తుంది।"),
    ("रचनात्मक लेखन के लिए कल्पनाशीलता और संवेदनशीलता का होना अनिवार्य गुण माना जाता है।", "సృజనాత్మక రచనకు ఊహాశక్తి మరియు సంవేదనాత్మకత ఉండటం అనివార్యమైన లక్షణాలు।"),
    ("खेलकूद न केवल शारीरिक स्वास्थ्य बल्कि टीम भावना और नेतृत्व क्षमता का भी विकास करते हैं।", "క్రీడలు కేవలం శారీరక ఆరోగ్యాన్నే కాకుండా జట్టు స్ఫూర్తిని మరియు నాయకత్వ పటిమను కూడా పెంచుతాయి।"),
    ("समय का प्रबंधन करने वाले व्यक्ति ही पेशेवर और व्यक्तिगत जीवन में सफलता प्राप्त करते हैं।", "సమయాన్ని సద్వినియోగం చేసుకునే వ్యక్తులు మాత్రమే వృత్తిపరమైన మరియు వ్యక్తిగత జీవితంలో విజయం సాధిస్తారు।"),
    ("सकारात्मक सोच मानसिक तनाव को कम करने और जीवन के प्रति उत्साह बढ़ाने में सहायक होती है।", "సానుకూల ఆలోచన మానసిక ఒత్తిడిని తగ్గించడానికి మరియు జీవితం పట్ల ఉత్సాహాన్ని పెంచడానికి సహాయపడుతుంది।"),
    ("आत्मविश्वास और कड़ी मेहनत का कोई विकल्प नहीं है जब बात सपनों को पूरा करने की हो।", "కలలను నెరవేర్చుకోవడంలో ఆత్మవిశ్వాసం మరియు కష్టపడటానికి మించిన ప్రత్యామ్నాయం లేదు।"),
    ("डिजिटल साक्षरता आज के युग में उतनी ही महत्वपूर्ण है जितनी कि पारंपरिक साक्षरता।", "అవగాహన కలిగిన పౌరుడు మాత్రమే ప్రజాస్వామ్య మూలాలను బలోపేతం చేయడంలో కీలక పాత్ర పోషిస్తాడు।"),
    ("एक जागरूक नागरिक ही लोकतंत्र की जड़ों को मजबूत बनाने में महत्वपूर्ण भूमिका निभाता है।", "సాంప్రదాయ అక్షరాస్యతతో పాటు డిజిటల్ అక్షరాస్యత కూడా నేటి యుగంలో అంతే ముఖ్యం।")
]

# --- EXECUTION ---
aligner = DeepAligner()
for i, (hi, te) in enumerate(COMPLEX_DATASET):
    links, score = aligner.align_complex(hi, te)
    print(f"\n--- SENTENCE {i+1} (Quality: {score:.4f}) ---")
    print(f"Hindi: {hi}\nTelugu: {te}\n")
    print(f"{'HINDI WORD':<20} | {'TELUGU WORD':<20} | {'CONFIDENCE'}")
    print("-" * 55)

    # Sort by sentence order (i) to see the flow
    for h_word, t_word, conf in sorted(links, key=lambda x: x[2], reverse=True):
        print(f"{h_word:<20} | {t_word:<20} | {conf:.5f}")

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]


--- SENTENCE 1 (Quality: 0.7758) ---
Hindi: भारतीय संविधान की प्रस्तावना में निहित धर्मनिरपेक्षता, समाजवाद और अखंडता के आदर्श राष्ट्र की एकता को सुदृढ़ करते हैं।
Telugu: భారత రాజ్యాంగ పీఠికలో పొందుపరచబడిన లౌకికవాదం, సామ్యవాదం మరియు అఖండత అనే ఆదర్శాలు దేశ ఐక్యతను బలోపేతం చేస్తాయి।

HINDI WORD           | TELUGU WORD          | CONFIDENCE
-------------------------------------------------------
भारतीय               | భారత                 | 1.00000
संविधान              | రాజ్యాంగ             | 1.00000
की                   | పీఠికలో              | 1.00000
निहित                | పొందుపరచబడిన         | 1.00000
समाजवाद              | సామ్యవాదం            | 1.00000
और                   | మరియు                | 1.00000
आदर्श                | ఆదర్శాలు             | 1.00000
राष्ट्र              | దేశ                  | 1.00000
एकता                 | ఐక్యతను              | 1.00000
सुदृढ़               | బలోపేతం              | 1.00000
करते                 | చేస్తాయి             | 1.00000
अखंडता    

In [ ]:
import torch
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from torch.nn.functional import softmax

class ResearchDeepAligner:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        # LaBSE is the state-of-the-art for cross-lingual word alignment
        self.model = SentenceTransformer('sentence-transformers/LaBSE').to(self.device)
        self.tokenizer = self.model.tokenizer

    def _get_vectors(self, tokens):
        inputs = self.tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.model[0].auto_model(**inputs, output_hidden_states=True)
            # Using Layer 8 for deep semantic features as per cross-lingual alignment research
            embeddings = outputs.hidden_states[8][0]

        word_ids = inputs.word_ids(batch_index=0)
        return torch.stack([embeddings[[j for j, w_id in enumerate(word_ids) if w_id == i]].mean(dim=0)
                            for i in range(len(tokens))])

    def align_and_evaluate(self, hi_text, te_text):
        # 1. Cleaning and Tokenization
        hi_toks = re.sub(r'[।\?\!\,\(\)"]', '', hi_text).strip().split()
        te_toks = re.sub(r'[।\?\!\,\(\)"]', '', te_text).strip().split()

        if not hi_toks or not te_toks: return None

        # 2. Vector Generation
        hi_vecs = self._get_vectors(hi_toks)
        te_vecs = self._get_vectors(te_toks)

        # 3. Double Softmax Alignment (Symmetry check)
        sim_mat = torch.matmul(hi_vecs, te_vecs.T)
        # Sharpness temperature of 12.0 filters out low-probability noise
        s_hi = softmax(sim_mat * 12.0, dim=1)
        s_te = softmax(sim_mat * 12.0, dim=0)
        mutual_conf = (s_hi * s_te).cpu().numpy()

        links = []
        hi_matched = set()
        te_matched = set()

        # 4. Extracting All Confident Links
        for i in range(len(hi_toks)):
            for j in range(len(te_toks)):
                conf = mutual_conf[i, j]
                if conf > 0.05: # Detection threshold for meaningful connection
                    links.append((hi_toks[i], te_toks[j], conf))
                    hi_matched.add(i)
                    te_matched.add(j)

        # 5. Feature: Identify Missed Links
        missed_hi = [hi_toks[i] for i in range(len(hi_toks)) if i not in hi_matched]
        missed_te = [te_toks[j] for j in range(len(te_toks)) if j not in te_matched]

        # 6. Quality Scoring
        avg_conf = np.mean([l[2] for l in links]) if links else 0
        coverage = len(links) / max(len(hi_toks), len(te_toks))
        q_score = (avg_conf + coverage) / 2

        # 7. Research-Based Categorization
        # GOOD (>= 0.6): High semantic density
        # MODERATE (0.4 - 0.6): Partial agreement/Agglutination noise
        # WRONG (< 0.4): Failed alignment
        if q_score >= 0.6:
            category = "GOOD"
        elif 0.4 <= q_score < 0.6:
            category = "MODERATE"
        else:
            category = "WRONG/WEAK"

        return {
            "links": sorted(links, key=lambda x: x[2], reverse=True),
            "score": q_score,
            "category": category,
            "missed_hi": missed_hi,
            "missed_te": missed_te
        }

# --- DATASET ---
# Using the 100 COMPLEX_DATASET provided previously
COMPLEX_DATASET = [
    # --- I. LEGAL & GOVERNANCE ---
    ("भारतीय संविधान की प्रस्तावना में निहित धर्मनिरपेक्षता, समाजवाद और अखंडता के आदर्श राष्ट्र की एकता को सुदृढ़ करते हैं।", "భారత రాజ్యాంగ పీఠికలో పొందుపరచబడిన లౌకికవాదం, సామ్యవాదం మరియు అఖండత అనే ఆదర్శాలు దేశ ఐక్యతను బలోపేతం చేస్తాయి।"),
    ("उच्चतम न्यायालय ने निर्णय दिया कि निजता का अधिकार संविधान के अनुच्छेद 21 के तहत जीवन और व्यक्तिगत स्वतंत्रता का अभिन्न अंग है।", "గోప్యత హక్కు రాజ్యాంగంలోని 21వ అధికరణం ప్రకారం జీవించే మరియు వ్యక్తిగత స్వేచ్ఛలో అంతర్భాగమని సుప్రీంకోర్టు తీర్పునిచ్చింది।"),
    ("द्विसदनीय विधायिका की संरचना यह सुनिश्चित करती है कि केंद्र और राज्यों के बीच शक्तियों का वितरण संतुलन में बना रहे।", "ద్విసభా శాసనసభ నిర్మాణం కేంద్రం మరియు రాష్ట్రాల మధ్య అధికారాల పంపిణీ సమతుల్యంగా ఉండేలా చూస్తుంది।"),
    ("चुनाव आयोग एक स्वायत्त संवैधानिक प्राधिकरण है जो भारत में संघ और राज्य चुनाव प्रक्रियाओं के प्रशासन के लिए उत्तरदायी है।", "ఎన్నికల సంఘం అనేది భారతదేశంలో కేంద్ర మరియు రాష్ట్ర ఎన్నికల ప్రక్రియల నిర్వహణకు బాధ్యత వహించే ఒక స్వయంప్రతిపత్తి కలిగిన రాజ్యాంగ సంస్థ।"),
    ("लोकपाल और लोकायुक्त अधिनियम का उद्देश्य सार्वजनिक पदाधिकारियों के विरुद्ध भ्रष्टाचार के आरोपों की निष्पक्ष जांच करना है।", "లోక్‌పాల్ మరియు లోకాయుక్త చట్టం యొక్క ఉద్దేశ్యం ప్రభుత్వ అధికారులపై వచ్చే అవినీతి ఆరోపణలపై నిష్పక్షపాతంగా విచారణ జరపడం।"),
    ("मौलिक कर्तव्यों का पालन करना प्रत्येक नागरिक का नैतिक उत्तरदायित्व है ताकि समाज में सद्भाव और अनुशासन बना रहे।", "సమాజంలో సామరస్యం మరియు క్రమశిక్షణను కాపాడటానికి ప్రతి పౌరుడు ప్రాథమిక విధులను పాటించడం నైతిక బాధ్యత।"),
    ("संसद के दोनों सदनों द्वारा पारित विधेयक को राष्ट्रपति की सहमति प्राप्त होने के बाद ही आधिकारिक कानून माना जाता है।", "పార్లమెంటు ఉభయ సభలు ఆమోదించిన బిల్లుకు రాష్ట్రపతి అంగీకారం లభించిన తర్వాతే అది అధికారిక చట్టంగా మారుతుంది।"),
    ("प्रशासनिक विकेंद्रीकरण का मुख्य उद्देश्य स्थानीय स्वशासन संस्थानों को सशक्त बनाना और जमीनी स्तर पर विकास सुनिश्चित करना है।", "పరిపాలనా వికేంద్రీకరణ యొక్క ప్రధాన ఉద్దేశ్యం స్థానిక స్వపరిపాలన సంస్థలను బలోపేతం చేయడం మరియు క్షేత్రస్థాయిలో అభివృద్ధిని నిర్ధారించడం।"),
    ("न्यायिक सक्रियता के माध्यम से अदालतें अक्सर कार्यपालिका को उनके संवैधानिक कर्तव्यों के प्रति सचेत करने का कार्य करती हैं।", "న్యాయవ్యవస్థ క్రియాశీలత ద్వారా కోర్టులు తరచుగా కార్యనిర్వాహక వ్యవస్థను వారి రాజ్యాంగ విధులను గుర్తు చేస్తూ ఉంటాయి।"),
    ("अन्तरराष्ट्रीय न्यायालय का मुख्य कार्य राज्यों के बीच कानूनी विवादों को सुलझाना और कानूनी प्रश्नों पर सलाह देना है।", "అంతర్జాతీయ న్యాయస్థానం యొక్క ప్రధాన విధి దేశాల మధ్య చట్టపరమైన వివాదాలను పరిష్కరించడం మరియు చట్టపరమైన ప్రశ్నలపై సలహాలు ఇవ్వడం।"),

    # --- II. SPACE & PHYSICS ---
    ("भारतीय अंतरिक्ष अनुसंधान संगठन ने मंगलयान मिशन के सफल प्रक्षेपण के साथ अंतरग्रहीय अन्वेषण में नया अध्याय लिखा।", "భారత అంతరిక్ష పరిశోధనా సంస్థ మంగళయాన్ మిషన్ విజయవంత ప్రయోగంతో అంతర గ్రహ పరిశోధనలో కొత్త అధ్యాయాన్ని లిఖించింది।"),
    ("क्वांटम भौतिकी के सिद्धांत बताते हैं कि सूक्ष्म स्तर पर कणों का व्यवहार शास्त्रीय भौतिकी के नियमों से भिन्न होता है।", "క్వాంటం భౌతికశాస్త్ర సూత్రాలు సూక్ష్మ స్థాయిలో కణాల ప్రవర్తన శాస్త్రీయ భౌతికశాస్త్ర నియమాల కంటే భిన్నంగా ఉంటుందని చెబుతాయి।"),
    ("पारिस्थितिक तंत्र में जैव विविधता की हानि खाद्य श्रृंखला के स्थायित्व को गंभीर रूप से असंतुलित कर सकती है।", "పర్యావరణ వ్యవస్థలో జీవవైవిధ్యం తగ్గడం వల్ల ఆహార గొలుసు యొక్క స్థిరత్వం తీవ్రంగా దెబ్బతింటుంది।"),
    ("वैश्विक तापन के कारण ग्लेशियरों का पिघलना समुद्र के स्तर में वृद्धि और तटीय क्षेत्रों के जलमग्न होने का कारण है।", "భూతాపం వల్ల హిమానీనదాలు కరగడం సముద్ర మట్టాలు పెరగడానికి మరియు తీర ప్రాంతాలు మునిగిపోవడానికి కారణం।"),
    ("प्रकाश की दोहरी प्रकृति यह सिद्ध करती है कि वह तरंग और कण दोनों के रूप में व्यवहार कर सकता है।", "కాంతి యొక్క ద్వంద్వ స్వభావం అది తరంగం మరియు కణం రెండింటి రూపంలో ప్రవర్తించగలదని నిరూపిస్తుంది।"),
    ("ब्लैक होल अंतरिक्ष में वह स्थान है जहाँ गुरुत्वाकर्षण इतना प्रबल होता है कि प्रकाश भी वहां से नहीं निकल सकता।", "బ్లాక్ హోల్ అనేది అంతరిక్షంలో గురుత్వాకర్షణ శక్తి ఎంత బలంగా ఉంటుందంటే అక్కడ నుండి కాంతి కూడా తప్పించుకోలేదు।"),
    ("सौरमंडल के ग्रहों की गति को समझने के लिए केप्लर के नियमों का अध्ययन करना खगोल विज्ञान में अनिवार्य है।", "సౌర వ్యవస్థలోని గ్రహాల గమనాన్ని అర్థం చేసుకోవడానికి కెప్లర్ నియమాలను అధ్యయనం చేయడం ఖగోళ శాస్త్రంలో తప్పనిసరి।"),
    ("परमाणु रिएक्टरों में नियंत्रित नाभिकीय विखंडन की प्रक्रिया के माध्यम से भारी मात्रा में ऊर्जा उत्पन्न की जाती है।", "అణు రియాక్టర్లలో నియంత్రిత కేంద్రక విచ్ఛిత్తి ప్రక్రియ ద్వారా భారీ మొత్తంలో ఇంధనం ఉత్పత్తి చేయబడుతుంది।"),
    ("ओजोन परत वायुमंडल के ऊपरी भाग में स्थित है जो हानिकारक पराबैंगनी विकिरण को पृथ्वी तक पहुँचने से रोकती है।", "ఓజోన్ పొర వాతావరణం పైభాగంలో ఉండి హానికరమైన అతినీలలోహిత వికిరణం భూమికి చేరకుండా అడ్డుకుంటుంది।"),
    ("आइंस्टीन का सापेक्षता का सिद्धांत स्थान और समय के बीच के संबंध को वैज्ञानिक रूप से पुनर्परिभाषित करता है।", "ఐన్‌స్టీన్ సాపేక్షత సిద్ధాంతం స్థలం మరియు కాలం మధ్య సంబంధాన్ని శాస్త్రీయంగా పునర్నిర్వచించింది।"),

    # --- III. TECHNOLOGY & AI ---
    ("कृत्रिम बुद्धिमत्ता के एल्गोरिदम डेटा के विशाल भंडार का विश्लेषण करके भविष्य के रुझानों की सटीक भविष्यवाणी करते हैं।", "కృత్రిమ మేధస్సు అల్గారిథమ్స్ భారీ డేటాను విశ్లేషించి భవిష్యత్తు ధోరణులను ఖచ్చితంగా అంచనా వేస్తాయి।"),
    ("ब्लॉकचेन तकनीक विकेंद्रीकृत बहीखाता प्रणाली के माध्यम से वित्तीय लेनदेन की सुरक्षा और पारदर्शिता सुनिश्चित करती है।", "బ్లాక్‌చైన్ సాంకేతికత వికేంద్రీకృత లెడ్జర్ వ్యవస్థ ద్వారా ఆర్థిక లావాదేవీల భద్రత మరియు పారదర్శకతను నిర్ధారిస్తుంది।"),
    ("वस्तु एवं सेवा कर (GST) के कार्यान्वयन ने भारत में अप्रत्यक्ष कर ढांचे को एकीकृत करके साझा बाजार बनाया है।", "వస్తు సేవల పన్ను (GST) అమలు భారతదేశంలో పరోక్ష పన్ను నిర్మాణాన్ని ఏకీకృతం చేయడం ద్వారా ఉమ్మడి మార్కెట్‌ను సృష్టించింది।"),
    ("सूक्ष्म, लघु और मध्यम उद्यम क्षेत्र भारतीय अर्थव्यवस्था की रीढ़ है जो ग्रामीण रोजगार सृजन में योगदान देता है।", "సూక్ష్మ, చిన్న మరియు మధ్య తరహా పరిశ్రమల రంగం భారత ఆర్థిక వ్యవస్థకు వెన్నెముక, ఇది గ్రామీణ ఉపాధి కల్పనలో తోడ్పడుతుంది।"),
    ("साइबर सुरक्षा की चुनौतियाँ निरंतर बढ़ती जा रही हैं क्योंकि डिजिटल बुनियादी ढाँचा अधिक जटिल होता जा रहा है।", "డిజిటల్ మౌలిక సదుపాయాలు మరింత సంక్లిష్టంగా మారుతున్న కొద్దీ సైబర్ భద్రతా సవాళ్లు నిరంతరం పెరుగుతున్నాయి।"),
    ("मशीन लर्निंग के मॉडल जितना अधिक डेटा प्राप्त करते हैं, उनकी सटीकता और प्रदर्शन में उतना ही सुधार होता है।", "మెషిన్ లెర్నింగ్ మోడల్స్ ఎంత ఎక్కువ డేటాను పొందితే, వాటి ఖచ్చితత్వం మరియు పనితీరు అంత మెరుగుపడుతుంది।"),
    ("क्लाउड कंप्यूटिंग व्यवसायों को भौतिक बुनियादी ढांचे में निवेश किए बिना बड़े पैमाने पर डेटा स्टोर करने की अनुमति देती है।", "క్లౌడ్ కంప్యూటింగ్ భౌతిక మౌలిక సదుపాయాలపై పెట్టుబడి పెట్టకుండానే భారీ డేటాను భద్రపరిచేందుకు వ్యాపారాలకు అనుమతిస్తుంది।"),
    ("इंटरनेट ऑफ थिंग्स (IoT) विभिन्न उपकरणों को एक नेटवर्क से जोड़कर स्मार्ट स्वचालन की सुविधा प्रदान करता है।", "ఇంటర్నెట్ ఆఫ్ థింగ్స్ (IoT) వివిధ పరికరాలను నెట్‌వర్క్‌తో అనుసంధానించడం ద్వారా స్మార్ట్ ఆటోమేషన్‌కు వీలు కల్పిస్తుంది।"),
    ("स्वदेशी विनिर्माण को बढ़ावा देने के लिए 'मेक इन इंडिया' पहल ने रक्षा उत्पादन क्षेत्र में निवेश को आकर्षित किया है।", "స్వదేశీ తయారీని ప్రోత్సహించడానికి 'మేక్ ఇన్ ఇండియా' చొరవ రక్షణ ఉత్పత్తి రంగంలో పెట్టుబడులను ఆకర్షించింది।"),
    ("आर्थिक मंदी के दौरान सरकारें अक्सर तरलता बढ़ाने के लिए ब्याज दरों को कम करने जैसे राजकोषीय उपाय करती हैं।", "ఆర్థిక మాంద్యం సమయంలో ప్రభుత్వం లిక్విడిటీ పెంచడానికి వడ్డీ రేట్లను తగ్గించడం వంటి చర్యలు తీసుకుంటుంది।"),

    # --- IV. PHILOSOPHY & HISTORY ---
    ("अद्वैत वेदांत दर्शन के अनुसार आत्मा और परमात्मा के बीच का भेद केवल अविद्या या अज्ञान के कारण प्रतीत होता है।", "అద్వైత వేదాంత దర్శనం ప్రకారం ఆత్మ మరియు పరమాత్మ మధ్య భేదం కేవలం అవిద్య లేదా అజ్ఞానం వల్లనే కనిపిస్తుంది।"),
    ("मौर्य साम्राज्य के शिलालेख राजा अशोक की धम्म नीति और लोक कल्याणकारी प्रशासन के जीवंत प्रमाण प्रस्तुत करते हैं।", "మౌర్య సామ్రాజ్య శిలాశాసనాలు అశోక చక్రవర్తి ధమ్మ నీతి మరియు ప్రజా సంక్షేమ పాలనకు సజీవ సాక్ష్యాలుగా నిలుస్తాయి।"),
    ("सांस्कृतिक बहुलवाद भारतीय समाज की वह विशेषता है जो विभिन्न भाषाओं और परंपराओं को एक सूत्र में पिरोती है।", "సాంస్కృతిక బహుళత్వం అనేది భారతీయ సమాజం యొక్క ప్రత్యేకత, ఇది వివిధ భాషలను మరియు సంప్రదాయాలను ఒకే త్రాడుపైకి తెస్తుంది।"),
    ("पुनर्जागरण काल ने कला और विज्ञान के क्षेत्र में तर्क और मानवतावाद को धार्मिक कट्टरता पर वरीयता दी।", "పునర్జీవన కాలం కళ మరియు విజ్ఞాన రంగాలలో మతపరమైన ఛాందసవాదం కంటే తర్కానికి మరియు మానవతావాదానికి ప్రాధాన్యతనిచ్చింది।"),
    ("प्राचीन भारतीय चिकित्सा पद्धति आयुर्वेद रोगों के उपचार के बजाय जीवनशैली के संतुलन पर अधिक बल देती है।", "ప్రాచీన భారతీయ వైద్య విధానం ఆయుర్వేదం వ్యాధుల చికిత్స కంటే జీవనశైలి సమతుల్యతపై ఎక్కువ మొగ్గు చూపుతుంది।"),
    ("बुद्ध के अष्टांगिक मार्ग का उद्देश्य मानवीय दुखों का अंत करना और निर्वाण की प्राप्ति के लिए मार्गदर्शन करना है।", "బుద్ధుని అష్టాంగ మార్గం మానవ దుఃఖాలను అంతం చేయడానికి మరియు నిర్వాణం సాధించడానికి మార్గదర్శకత్వం చేస్తుంది।"),
    ("सिंधु घाटी सभ्यता की शहरी योजना और जल निकासी प्रणाली उस समय की उन्नत तकनीकी समझ को दर्शाती है।", "సింధు లోయ నాగరికత యొక్క పట్టణ ప్రణాళిక మరియు డ్రైనేజీ వ్యవస్థ అప్పటి అధునాతన సాంకేతిక అవగాహనను ప్రతిబింబిస్తాయి।"),
    ("भक्ति आंदोलन ने जातिगत बंधनों को तोड़कर ईश्वर के प्रति व्यक्तिगत समर्पण और प्रेम के मार्ग को लोकप्रिय बनाया।", "భక్తి ఉద్యమం కుల బంధాలను తెంచి భగవంతుని పట్ల వ్యక్తిగత అంకితభావం మరియు ప్రేమ మార్గాన్ని ప్రాచుర్యంలోకి తెచ్చింది।"),
    ("गांधीवादी दर्शन में सत्य और अहिंसा को केवल आदर्श नहीं बल्कि सामाजिक परिवर्तन के अस्त्र के रूप में देखा गया है।", "గాంధేయ దర్శనంలో సత్యం మరియు అహింస కేవలం ఆదర్శాలు మాత్రమే కాదు, సామాజిక మార్పుకు అస్త్రాలుగా చూడబడ్డాయి।"),
    ("मुगल वास्तुकला में भारतीय और फारसी शैलियों का अद्भुत समन्वय ताजमहल जैसी भव्य इमारतों में परिलक्षित होता है।", "మొఘల్ వాస్తుశిల్పంలో భారతీయ మరియు పర్షియన్ శైలుల అద్భుత సమన్వయం తాజ్ మహల్ వంటి భవనాలలో కనిపిస్తుంది।"),

    # --- V. EDUCATION & ETHICS (41-60) ---
    ("नई शिक्षा नीति का उद्देश्य छात्रों में रटने की प्रवृत्ति के बजाय आलोचनात्मक सोच और रचनात्मकता विकसित करना है।", "కొత్త విద్యా విధానం విద్యార్థులలో బట్టీ పట్టే ధోరణి కంటే విమర్శనాత్మక ఆలోచన మరియు సృజనాత్మకతను పెంపొందించడం లక్ష్యంగా పెట్టుకుంది।"),
    ("समावेशी विकास का अर्थ है कि आर्थिक प्रगति का लाभ समाज के सबसे अंतिम पायदान पर खड़े व्यक्ति तक पहुँचे।", "సమ్మిళిత వృద్ధి అంటే ఆర్థిక పురోగతి యొక్క ప్రయోజనాలు సమాజంలోని చివరి వ్యక్తికి కూడా అందడం।"),
    ("नैतिक मूल्य ही वह आधार हैं जिस पर एक स्वस्थ और न्यायपूर्ण समाज की नींव रखी जा सकती है।", "నైతిక విలువల ప్రాతిపదికన మాత్రమే ఆరోగ్యకరమైన మరియు న్యాయమైన సమాజ పునాది నిర్మించబడుతుంది।"),
    ("तकनीकी शिक्षा के साथ-साथ मानवीय संवेदनाओं का विकास होना भी आधुनिक युग की एक अनिवार्य आवश्यकता है।", "సాంకేతిక విద్యతో పాటు మానవీయ संवेदनाల అభివృద్ధి కూడా ఆధునిక యుగంలో అనివార్యమైన అవసరం।"),
    ("पर्यावरण संरक्षण के प्रति जागरूकता फैलाना केवल सरकार का नहीं बल्कि प्रत्येक नागरिक का मौलिक कर्तव्य है।", "పర్యావరణ పరిరక్షణపై అవగాహన కల్పించడం కేవలం ప్రభుత్వానిదే కాదు, ప్రతి పౌరుడి ప్రాథమిక విధి।"),
    ("महिलाओं के आर्थिक सशक्तिकरण से न केवल परिवार की स्थिति सुधरती है बल्कि राष्ट्र का विकास भी तेज होता है।", "మహిళల ఆర్థిక సాధికారతతో కుటుంబ పరిస్థితి మెరుగుపడటమే కాకుండా దేశ అభివృద్ధి కూడా వేగవంతమవుతుంది।"),
    ("भ्रष्टाचार मुक्त समाज के निर्माण के लिए पारदर्शिता और जवाबदेही को हर स्तर पर लागू करना अनिवार्य है।", "అవినీతి రహిత సమాజం కోసం పారదర్శకత మరియు జవాబుదారీతనాన్ని ప్రతి స్థాయిలో అమలు చేయడం తప్పనిసరి।"),
    ("श्रम की गरिमा का सम्मान करना समाज के सर्वांगीण विकास के लिए एक महत्वपूर्ण सांस्कृतिक मूल्य है।", "శ్రమ గౌరవాన్ని గౌరవించడం సమాజ సర్వతోముఖాభివృద్ధికి ఒక ముఖ్యమైన సాంస్కృతిక విలువ।"),
    ("प्राकृतिक संसाधनों का विवेकपूर्ण उपयोग भविष्य की पीढ़ियों की आवश्यकताओं को सुरक्षित करने के लिए जरूरी है।", "భవిష్యత్ తరాల అవసరాలను సురక్షితం చేయడానికి సహజ వనరుల వివేకవంతమైన ఉపయోగం అవసరం।"),
    ("वैश्विक शांति की स्थापना के लिए विभिन्न राष्ट्रों के बीच सांस्कृतिक आदान-प्रदान और संवाद आवश्यक है।", "ప్రపంచ శాంతి స్థాపన కోసం వివిధ దేశాల మధ్య సాంస్కృతిక మార్పిడి మరియు సంభాషణ అవసరం।"),

    # --- VI. MEDICINE & BIOLOGY (51-70) ---
    ("प्रतिरक्षा प्रणाली शरीर को बाहरी बैक्टीरिया और वायरस के संक्रमण से बचाने वाली एक जटिल सुरक्षा व्यवस्था है।", "రోగనిరోధక వ్యవస్థ అనేది శరీరాన్ని బయటి బ్యాక్టీరియా మరియు వైరస్ల నుండి రక్షించే ఒక సంక్లిష్ట రక్షణ వ్యవస్థ।"),
    ("कैंसर अनुसंधान के क्षेत्र में इम्यूनोथेरेपी को एक क्रांतिकारी उपचार पद्धति के रूप में देखा जा रहा है।", "క్యాన్సర్ పరిశోధనా రంగంలో ఇమ్యునోథెరపీ ఒక విప్లవాత్మక చికిత్సా విధానంగా పరిగణించబడుతోంది।"),
    ("कोशिका विभाजन की अनियंत्रित प्रक्रिया ही ट्यूमर के विकास का मुख्य जैविक कारण मानी जाती है।", "కణ విభజన యొక్క అనియంత్రిత ప్రక్రియే ట్యూమర్ అభివృద్ధికి ప్రధాన జీవసంబంధ కారణం।"),
    ("हृदय रोगों के जोखिम को कम करने के लिए संतुलित आहार और नियमित व्यायाम का पालन करना अनिवार्य है।", "గుండె జబ్బుల ప్రమాదాన్ని తగ్గించడానికి సమతుల్య ఆహారం మరియు క్రమం తప్పకుండా వ్యాయామం చేయడం తప్పనిసరి।"),
    ("डीएनए अनुक्रमण तकनीक ने वंशानुगत बीमारियों की पहचान और उनके निवारण में अभूतपूर्व प्रगति की है।", "DNA సీక్వెన్సింగ్ సాంకేతికత వంశపారంపర్య వ్యాధుల గుర్తింపు మరియు నివారణలో అపూర్వమైన పురోగతిని సాధించింది।"),
    ("तंत्रिका तंत्र मस्तिष्क और शरीर के अन्य अंगों के बीच सूचनाओं के संचरण का समन्वय करता है।", "నరాల వ్యవస్థ మెదడు మరియు శరీరంలోని ఇతర భాగాల మధ్య సమాచార ప్రసారాన్ని సమన్వయం చేస్తుంది।"),
    ("एंटीबायोटिक दवाओं का अत्यधिक उपयोग भविष्य में दवाओं के प्रति प्रतिरोधक क्षमता विकसित होने का खतरा पैदा करता है।", "యాంటీబయాటిక్స్ అధికంగా వాడటం వల్ల భవిష్యత్తులో ఔషధ నిరోధకత పెరిగే ప్రమాదం ఉంది।"),
    ("रक्त शर्करा के स्तर को नियंत्रित करने के लिए अग्न्याशय द्वारा इंसुलिन हार्मोन का स्राव अत्यंत महत्वपूर्ण है।", "రక్తంలో చక్కెర స్థాయిలను నియంత్రించడానికి ప్యాంక్రియాస్ ఇన్సులిన్ హార్మోన్‌ను విడుదల చేయడం చాలా ముఖ్యం।"),
    ("विटामिन डी की कमी से हड्डियों की मजबूती प्रभावित होती है और ऑस्टियोपोरोसिस का खतरा बढ़ जाता है।", "విటమిన్ డి లోపం వల్ల ఎముకల బలం తగ్గుతుంది మరియు ఆస్టియోపోరోసిస్ ప్రమాదం పెరుగుతుంది।"),
    ("मानसिक स्वास्थ्य के प्रति सामाजिक दृष्टिकोण में बदलाव लाना चिकित्सा विज्ञान की एक बड़ी चुनौती है।", "మానసిక ఆరోగ్యం పట్ల సామాజిక దృక్పథంలో మార్పు తీసుకురావడం వైద్య విజ్ఞానానికి ఒక పెద్ద సవాలు।"),

    # --- VII. INFRASTRUCTURE & URBANIZATION (71-85) ---
    ("स्मार्ट सिटी मिशन का लक्ष्य शहरी क्षेत्रों में डिजिटल तकनीक के माध्यम से जीवन की गुणवत्ता में सुधार करना है।", "స్మార్ట్ సిటీ మిషన్ లక్ష్యం పట్టణ ప్రాంతాల్లో డిజిటల్ సాంకేతికత ద్వారా జీవన ప్రమాణాలను మెరుగుపరచడం।"),
    ("मेट्रो रेल परियोजनाओं ने बड़े शहरों में सार्वजनिक परिवहन की सुविधा और दक्षता को पूरी तरह से बदल दिया है।", "మెట్రో రైలు ప్రాజెక్టులు ప్రధాన నగరాల్లో ప్రజా రవాణా సౌకర్యాన్ని మరియు సామర్థ్యాన్ని పూర్తిగా మార్చేశాయి।"),
    ("सतत शहरीकरण के लिए अपशिष्ट प्रबंधन और जल पुनर्चक्रण प्रणालियों का विकास करना अनिवार्य हो गया है।", "సుస్థిర పట్టణీకరణ కోసం వ్యర్థాల నిర్వహణ మరియు నీటి పునర్వినియోగ వ్యవస్థల అభివృద్ధి అనివార్యమైంది।"),
    ("ग्रामीण बुनियादी ढांचे में सुधार से कृषि उत्पादों के विपणन और किसानों की आय में वृद्धि सुनिश्चित होती है।", "గ్రామీణ మౌలిక సదుపాయాల మెరుగుదల ద్వారా వ్యవసాయ ఉత్పత్తుల మార్కెటింగ్ మరియు రైతుల ఆదాయం పెరుగుతుంది।"),
    ("बंदरगाहों के आधुनिकीकरण से अंतरराष्ट्रीय व्यापार में लगने वाले समय और लागत को काफी कम किया जा सकता है।", "ఓడరేవుల ఆధునీకరణ ద్వారా అంతర్జాతీయ వాణిజ్యానికి పట్టే సమయాన్ని మరియు ఖర్చును గణనీయంగా తగ్గించవచ్చు।"),
    ("ऊर्जा सुरक्षा सुनिश्चित करने के लिए सौर और पवन ऊर्जा जैसे नवीकरणीय स्रोतों पर निर्भरता बढ़ानी होगी।", "ఇంధన భద్రతను నిర్ధారించడానికి సౌర మరియు పవన విద్యుత్ వంటి పునరుత్పాదక వనరులపై ఆధారపడటం పెంచాలి।"),
    ("पहाड़ी क्षेत्रों में सड़कों का निर्माण करते समय पारिस्थितिक संतुलन का ध्यान रखना एक बड़ी इंजीनियरिंग चुनौती है।", "కొండ ప్రాంతాలలో రోడ్ల నిర్మాణం చేపట్టేటప్పుడు పర్యావరణ సమతుల్యతను కాపాడటం ఒక పెద్ద ఇంజనీరింగ్ సవాలు।"),
    ("आवास योजनाओं का मुख्य उद्देश्य आर्थिक रूप से कमजोर वर्गों के लिए किफायती घर उपलब्ध कराना है।", "గృహనిర్మాణ పథకాల ప్రధాన ఉద్దేశ్యం ఆర్థికంగా వెనుకబడిన వర్గాలకు తక్కువ ధరకే ఇళ్లు అందించడం।"),
    ("डिजिटल कनेक्टिविटी ने दूरदराज के क्षेत्रों में शिक्षा और स्वास्थ्य सेवाओं की पहुँच को सुगम बना दिया है।", "డిజిటల్ కనెక్టివిటీ దూరప్రాంతాల్లో విద్య మరియు ఆరోగ్య సేవలు అందుబాటులోకి రావడాన్ని సులభతరం చేసింది।"),
    ("राष्ट्रीय राजमार्गों का नेटवर्क राज्यों के बीच वाणिज्यिक गतिविधियों और आर्थिक विकास की गति को बढ़ाता है।", "జాతీయ రహదారుల నెట్‌వర్క్ రాష్ట్రాల మధ్య వాణిజ్య కార్యకలాపాలను మరియు ఆర్థిక అభివృద్ధి వేగాన్ని పెంచుతుంది।"),

    # --- VIII. GENERAL COMPLEX TOPICS (86-100) ---
    ("वैश्वीकरण ने दुनिया को एक वैश्विक गांव में बदल दिया है जहाँ सूचनाओं का आदान-प्रदान तात्कालिक है।", "ప్రపంచీకరణ ప్రపంచాన్ని ఒక ప్రపంచ గ్రామంగా మార్చింది, ఇక్కడ సమాచార మార్పిడి తక్షణమే జరుగుతుంది।"),
    ("साहित्य समाज का दर्पण होता है जो प्रचलित सामाजिक मूल्यों और विसंगतियों को शब्दों में पिरोता है।", "సాహిత్యం సమాజానికి దర్పణం, ఇది ప్రబలమైన సామాజిక విలువలను మరియు వైరుధ్యాలను అక్షరబద్ధం చేస్తుంది।"),
    ("अनुशासन और निरंतरता ही वह कुंजी है जो किसी भी कठिन लक्ष्य को प्राप्त करने में सहायक होती है।", "క్రమశిక్షణ మరియు నిరంతర కృషి మాత్రమే ఏదైనా కష్టమైన లక్ష్యాన్ని సాధించడంలో సహాయపడతాయి।"),
    ("युवा शक्ति किसी भी राष्ट्र की सबसे बड़ी संपत्ति है जिसका सही दिशा में उपयोग करना आवश्यक है।", "యువశక్తి ఏ దేశానికైనా అతిపెద్ద ఆస్తి, దానిని సరైన దిశలో ఉపయోగించడం చాలా అవసరం।"),
    ("धैर्य और संयम विपत्ति के समय व्यक्ति के चरित्र की सबसे बड़ी परीक्षा लेते हैं।", "ధైర్యం మరియు సంయమనం కష్టకాలంలో వ్యక్తి యొక్క స్వభావాన్ని పరీక్షించే ప్రధాన అంశాలు।"),
    ("इतिहास हमें अतीत की गलतियों से सीखने और भविष्य को बेहतर बनाने की प्रेरणा देता है।", "చరిత్ర మనకు గత తప్పిదాల నుండి నేర్చుకోవడానికి మరియు భవిష్యత్తును మెరుగుపరుచుకోవడానికి స్ఫూర్తినిస్తుంది।"),
    ("पर्यटन क्षेत्र न केवल विदेशी मुद्रा अर्जित करता है बल्कि विभिन्न संस्कृतियों के बीच सेतु का कार्य करता है।", "పర్యాటక రంగం విదేశీ మారక ద్రవ్యాన్ని ఆర్జించడమే కాకుండా వివిధ సంస్కృతుల మధ్య వంతెనలా పనిచేస్తుంది।"),
    ("विज्ञान और अध्यात्म के बीच का संतुलन ही मनुष्य को पूर्णता की ओर ले जा सकता है।", "విజ్ఞానం మరియు ఆధ్యాత్మికత మధ్య సమతుల్యత మాత్రమే మనిషిని పరిపూర్ణత వైపు నడిపిస్తుంది।"),
    ("रचनात्मक लेखन के लिए कल्पनाशीलता और संवेदनशीलता का होना अनिवार्य गुण माना जाता है।", "సృజనాత్మక రచనకు ఊహాశక్తి మరియు సంవేదనాత్మకత ఉండటం అనివార్యమైన లక్షణాలు।"),
    ("खेलकूद न केवल शारीरिक स्वास्थ्य बल्कि टीम भावना और नेतृत्व क्षमता का भी विकास करते हैं।", "క్రీడలు కేవలం శారీరక ఆరోగ్యాన్నే కాకుండా జట్టు స్ఫూర్తిని మరియు నాయకత్వ పటిమను కూడా పెంచుతాయి।"),
    ("समय का प्रबंधन करने वाले व्यक्ति ही पेशेवर और व्यक्तिगत जीवन में सफलता प्राप्त करते हैं।", "సమయాన్ని సద్వినియోగం చేసుకునే వ్యక్తులు మాత్రమే వృత్తిపరమైన మరియు వ్యక్తిగత జీవితంలో విజయం సాధిస్తారు।"),
    ("सकारात्मक सोच मानसिक तनाव को कम करने और जीवन के प्रति उत्साह बढ़ाने में सहायक होती है।", "సానుకూల ఆలోచన మానసిక ఒత్తిడిని తగ్గించడానికి మరియు జీవితం పట్ల ఉత్సాహాన్ని పెంచడానికి సహాయపడుతుంది।"),
    ("आत्मविश्वास और कड़ी मेहनत का कोई विकल्प नहीं है जब बात सपनों को पूरा करने की हो।", "కలలను నెరవేర్చుకోవడంలో ఆత్మవిశ్వాసం మరియు కష్టపడటానికి మించిన ప్రత్యామ్నాయం లేదు।"),
    ("डिजिटल साक्षरता आज के युग में उतनी ही महत्वपूर्ण है जितनी कि पारंपरिक साक्षरता।", "అవగాహన కలిగిన పౌరుడు మాత్రమే ప్రజాస్వామ్య మూలాలను బలోపేతం చేయడంలో కీలక పాత్ర పోషిస్తాడు।"),
    ("एक जागरूक नागरिक ही लोकतंत्र की जड़ों को मजबूत बनाने में महत्वपूर्ण भूमिका निभाता है।", "సాంప్రదాయ అక్షరాస్యతతో పాటు డిజిటల్ అక్షరాస్యత కూడా నేటి యుగంలో అంతే ముఖ్యం।")
]

# --- EXECUTION ENGINE ---
def run_full_evaluation():
    aligner = ResearchDeepAligner()
    stats = {"GOOD": 0, "MODERATE": 0, "WRONG/WEAK": 0}

    print(f"🚀 Analyzing {len(COMPLEX_DATASET)} complex sentence pairs...\n")

    for i, (hi, te) in enumerate(COMPLEX_DATASET):
        res = aligner.align_and_evaluate(hi, te)
        if not res: continue

        stats[res["category"]] += 1

        print(f"--- SENTENCE {i+1} | {res['category']} (Score: {res['score']:.4f}) ---")

        # Display Missed Words
        if res["missed_hi"]: print(f"   🚫 Missed Hindi: {res['missed_hi']}")
        if res["missed_te"]: print(f"   🚫 Missed Telugu: {res['missed_te']}")

        # Display Full Alignment Mapping
        print(f"   🔗 Mappings:")
        for h, t, c in res["links"]:
            print(f"      {h:<15} ↔ {t:<15} (Conf: {c:.4f})")
        print("-" * 60)

    # --- FINAL REPORT ---
    total = sum(stats.values())
    print("\n" + "="*45)
    print(f"📊 FINAL ALIGNMENT ANALYTICS")
    print(f"GOOD (High Quality):    {stats['GOOD']} ({(stats['GOOD']/total)*100:.1f}%)")
    print(f"MODERATE (Partial):     {stats['MODERATE']} ({(stats['MODERATE']/total)*100:.1f}%)")
    print(f"WRONG (Mismatch):       {stats['WRONG/WEAK']} ({(stats['WRONG/WEAK']/total)*100:.1f}%)")
    print("="*45)

if __name__ == "__main__":
    run_full_evaluation()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Analyzing 85 complex sentence pairs...

--- SENTENCE 1 | GOOD (Score: 0.7895) ---
   🚫 Missed Hindi: ['प्रस्तावना', 'में', 'धर्मनिरपेक्षता', 'अखंडता', 'के', 'की', 'को', 'हैं']
   🚫 Missed Telugu: ['లౌకికవాదం', 'అఖండత', 'అనే']
   🔗 Mappings:
      भारतीय          ↔ భారత            (Conf: 1.0000)
      संविधान         ↔ రాజ్యాంగ        (Conf: 1.0000)
      की              ↔ పీఠికలో         (Conf: 1.0000)
      निहित           ↔ పొందుపరచబడిన    (Conf: 1.0000)
      समाजवाद         ↔ సామ్యవాదం       (Conf: 1.0000)
      और              ↔ మరియు           (Conf: 1.0000)
      आदर्श           ↔ ఆదర్శాలు        (Conf: 1.0000)
      राष्ट्र         ↔ దేశ             (Conf: 1.0000)
      एकता            ↔ ఐక్యతను         (Conf: 1.0000)
      सुदृढ़          ↔ బలోపేతం         (Conf: 1.0000)
      करते            ↔ చేస్తాయి        (Conf: 1.0000)
------------------------------------------------------------
--- SENTENCE 2 | GOOD (Score: 0.7826) ---
   🚫 Missed Hindi: ['न्यायालय', 'ने', 'दिया', 'कि